<a href="https://colab.research.google.com/github/WaleedAlsaedi/phishing-transfer-learning/blob/main/Adaptive_Transfer_Learning_for_Robust_Phishing_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Adaptive Transfer-Learning for Robust Phishing Detection

## Week 1 - Dataset Preprocessing

**Dataset** **1: Phishing Site URLs - Preprocessing**

In [ ]:
import pandas as pd

# Load URL dataset
df_url = pd.read_csv("phishing_site_urls.csv")

print("URL Dataset Shape:", df_url.shape)
print("\nURL Dataset Columns:")
print(df_url.columns)

df_url.head()

**Inspection**

In [ ]:
# Check label distribution
print(df_url['Label'].value_counts())

# Check missing values
print("\nMissing values:")
print(df_url.isna().sum())

# Check duplicate rows
print("\nDuplicate rows:", df_url.duplicated().sum())

**Cleaning**

In [ ]:
# Remove duplicate rows
df_url_clean = df_url.drop_duplicates()

print("Before removing duplicates:", df_url.shape)
print("After removing duplicates:", df_url_clean.shape)

**Encoding**

In [ ]:
# Encode labels: bad = 1, good = 0
df_url_clean['Label_encoded'] = df_url_clean['Label'].map({
    'good': 0,
    'bad': 1
})

# Check encoding
print(df_url_clean[['Label', 'Label_encoded']].head())
print("\nEncoded label distribution:")
print(df_url_clean['Label_encoded'].value_counts())

**Feature Extraction (URL Lexical Analysis)**

In the project plan, URL lexical analysis is applied to extract numerical features
from raw URLs. This step converts text-based URLs into structured numerical features
that can be used by traditional machine learning classifiers.

In [ ]:
from collections import Counter
import numpy as np

def extract_url_features(url):
    """Extract 16 lexical features from a URL string.

    Feature names are aligned with the Web Page Phishing dataset's column
    naming convention (n_dots, n_hypens, ...) to enable cross-dataset transfer
    learning in Approach 2 (Section 5.2). Where a feature has no Web Page
    counterpart, we keep the URL-specific name (domain_length, has_https, etc.).
    """
    features = {}

    # 1. URL length (shared name with Web Page dataset)
    features['url_length'] = len(url)

    # 2. Domain - for URLs without protocol, the domain is everything before
    #    the first '/'.  This matches the structure of the Phishing Site URLs
    #    dataset, where ~99.8% of URLs are stored without an http(s):// prefix.
    if '://' in url:
        rest = url.split('://', 1)[1]
    else:
        rest = url
    slash_idx = rest.find('/')
    if slash_idx == -1:
        domain = rest
        path_and_query = ''
    else:
        domain = rest[:slash_idx]
        path_and_query = rest[slash_idx:]

    features['domain_length'] = len(domain)

    # 3. Path length - strip query string (?...) and fragment (#...) so the
    #    feature represents only the resource path, not the entire URL tail.
    path_only = path_and_query
    for sep in ('?', '#'):
        if sep in path_only:
            path_only = path_only[:path_only.find(sep)]
    features['path_length'] = len(path_only)

    # 4-9. Character-count features.  Names are aligned with the Web Page
    #      dataset (n_dots, n_hypens, n_underline, n_slash, n_questionmark,
    #      n_and).  For Approach 2, these names will allow set-intersection
    #      to identify them as shared semantic features.
    features['n_dots']         = url.count('.')
    features['n_hypens']       = url.count('-')   # 'hypens' typo preserved to match Web Page dataset
    features['n_underline']    = url.count('_')
    features['n_slash']        = url.count('/')
    features['n_questionmark'] = url.count('?')
    features['n_and']          = url.count('&')

    # 10. n_at - count of '@' characters.  Web Page dataset uses count, so we
    #     match that convention (rather than a binary indicator).
    features['n_at'] = url.count('@')

    # 11. Has HTTPS prefix
    features['has_https'] = 1 if url.lower().startswith('https') else 0

    # 12. Number of digits in URL
    features['num_digits'] = sum(c.isdigit() for c in url)

    # 13. Number of special characters (URL-specific feature)
    special_chars = set('!#$%^&*()+=[]{}|;:,<>~`')
    features['num_special_chars'] = sum(c in special_chars for c in url)

    # 14. Number of subdomains in the domain
    features['num_subdomains'] = domain.count('.') - 1 if domain.count('.') > 1 else 0

    # 15. Has IP address as domain (e.g., '171.244.8.31/...').
    #     Strip optional ':port' before testing for purely numeric octets.
    domain_for_ip = domain.split(':')[0]
    features['has_ip'] = 1 if domain_for_ip.replace('.', '').isdigit() and domain_for_ip.count('.') == 3 else 0

    # 16. URL entropy (randomness of character distribution).
    if len(url) > 0:
        counts = Counter(url)
        probs = [count / len(url) for count in counts.values()]
        features['url_entropy'] = -sum(p * np.log2(p) for p in probs if p > 0)
    else:
        features['url_entropy'] = 0

    return features


# Apply feature extraction to all URLs
print("Extracting features from URLs... (this may take a few minutes)")
url_features = df_url_clean['URL'].apply(extract_url_features)
df_url_features = pd.DataFrame(url_features.tolist())

# Combine features with labels
df_url_features['Label_encoded'] = df_url_clean['Label_encoded'].values

print("\n Feature extraction complete!")
print("New feature set shape:", df_url_features.shape)
print("\nExtracted features (16 in total, named to align with Web Page dataset where applicable):")
print(df_url_features.columns.tolist())
print("\nSample of extracted features:")
df_url_features.head()


**Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Features and label (using extracted numerical features)
X_url = df_url_features.drop(columns=['Label_encoded'])
y_url = df_url_features['Label_encoded']

# Use a positional index split so the SAME indices can be reused later by
# DistilBERT in Week 4.  This guarantees that the DistilBERT test set contains
# exactly the same URLs whose lexical features are in the baseline test set,
# enabling a strict apples-to-apples comparison in Section 4.1.11.
positions = np.arange(len(X_url))
train_pos, test_pos = train_test_split(
    positions,
    test_size=0.2,
    random_state=42,
    stratify=y_url
)

X_url_train = X_url.iloc[train_pos].reset_index(drop=True)
X_url_test  = X_url.iloc[test_pos].reset_index(drop=True)
y_url_train = y_url.iloc[train_pos].reset_index(drop=True)
y_url_test  = y_url.iloc[test_pos].reset_index(drop=True)

# Keep the URL split indices in memory so Week 4 (DistilBERT) reuses them
url_train_pos = train_pos
url_test_pos  = test_pos

print("Train set size:", X_url_train.shape)
print("Test set size:",  X_url_test.shape)

print("\nTrain label distribution:")
print(y_url_train.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_url_test.value_counts(normalize=True))


Dataset 1: Phishing Site URLs - Preprocessing Summary (Week 1)

- The dataset was inspected to understand its structure and label distribution.
- Duplicate URLs were identified and removed to improve data quality.
- Labels were encoded into a binary format (0 = legitimate, 1 = phishing).
- **URL lexical analysis was applied to extract 16 numerical features from raw URLs, including URL length, domain length, character counts, entropy, and structural indicators.**
- The cleaned dataset was split into stratified training and testing sets (80/20).

This completes the dataset preprocessing stage for the URL-based phishing dataset.

**Dataset 2: Web Page Phishing Dataset - Preprocessing**

In [ ]:
import pandas as pd

df_web = pd.read_csv("web-page-phishing.csv")

print("Web dataset shape:", df_web.shape)
print("\nWeb dataset columns:")
print(df_web.columns)

df_web.head()

**Inspection**

In [ ]:
# Label distribution
print(df_web['phishing'].value_counts())

# Missing values
print("\nMissing values:")
print(df_web.isna().sum())

# Duplicate rows
print("\nDuplicate rows:", df_web.duplicated().sum())

**Cleaning**

In [ ]:
# Remove duplicate rows from web dataset
df_web_clean = df_web.drop_duplicates()

print("Before removing duplicates:", df_web.shape)
print("After removing duplicates:", df_web_clean.shape)

**Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_web = df_web_clean.drop(columns=['phishing'])
y_web = df_web_clean['phishing']

X_web_train, X_web_test, y_web_train, y_web_test = train_test_split(
    X_web,
    y_web,
    test_size=0.2,
    random_state=42,
    stratify=y_web
)

print("Train set:", X_web_train.shape)
print("Test set:", X_web_test.shape)

print("\nTrain label distribution:")
print(y_web_train.value_counts(normalize=True))

print("\nTest label distribution:")
print(y_web_test.value_counts(normalize=True))

Dataset 2: Web Page Phishing Dataset - Preprocessing Summary (Week 1)

- The dataset was examined to understand its structure and feature set, which consists of numerical URL-based characteristics.
- No label encoding was required, as all features and the target variable were already represented in numeric form.
- Missing value checks confirmed that the dataset contained no null entries.
- A large number of duplicate rows were identified and removed to ensure data quality.
- The cleaned dataset was split into stratified training and testing sets (80/20) to preserve class distribution.

This completes the preprocessing stage for the web page based phishing dataset.

## Week 2 - Baseline Machine Learning Model

In this section, baseline machine learning models are implemented using both the cleaned Web Page Phishing dataset and the URL Features dataset to establish initial performance benchmarks.

---

### 2.1 Baseline Models - Web Page Phishing Dataset

**Logistic Regression**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Train model
baseline_lr = LogisticRegression(max_iter=1000)
baseline_lr.fit(X_web_train, y_web_train)

# Predict on test set
y_pred_lr = baseline_lr.predict(X_web_test)
y_prob_lr = baseline_lr.predict_proba(X_web_test)[:, 1]

# Evaluate
print(" Logistic Regression - Baseline Results (Web Page Dataset)")
print("Accuracy:", accuracy_score(y_web_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_web_test, y_prob_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_web_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_web_test, y_pred_lr))

**Confusion Matrix** **(Web Page Dataset)** - **Logistic Regression**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_web_test, y_pred_lr)

plt.figure(figsize=(6,4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Legitimate (0)", "Phishing (1)"],
    yticklabels=["Legitimate (0)", "Phishing (1)"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Web Page Dataset) - Logistic Regression Baseline")
plt.tight_layout()
plt.show()

# **Compare with different models**

**Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

# 1) Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_web_train, y_web_train)

# 2) Predict
y_pred_rf = rf_model.predict(X_web_test)
y_prob_rf = rf_model.predict_proba(X_web_test)[:, 1]

# 3) Evaluate
print(" Random Forest - Baseline Results (Web Page Dataset)")
print("Accuracy:", accuracy_score(y_web_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_web_test, y_prob_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_web_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_web_test, y_pred_rf))

**Confusion Matrix** **(Web Page Dataset)** - **Random Forest**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_rf = confusion_matrix(y_web_test, y_pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Web Page Dataset) - Random Forest Baseline")
plt.tight_layout()
plt.show()

**KNN**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# 1) Build KNN pipeline (scaling + model)
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

# 2) Train
knn_model.fit(X_web_train, y_web_train)

# 3) Predict
y_pred_knn = knn_model.predict(X_web_test)
y_prob_knn = knn_model.predict_proba(X_web_test)[:, 1]

# 4) Evaluate
print(" KNN - Baseline Results (Web Page Dataset)")
print("Accuracy:", accuracy_score(y_web_test, y_pred_knn))
print("ROC-AUC:", roc_auc_score(y_web_test, y_prob_knn))
print("\nConfusion Matrix:\n", confusion_matrix(y_web_test, y_pred_knn))
print("\nClassification Report:\n", classification_report(y_web_test, y_pred_knn))

**Confusion Matrix** **(Web Page Dataset)** **- KNN**

---

In [ ]:
cm_knn = confusion_matrix(y_web_test, y_pred_knn)

plt.figure(figsize=(6,4))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Web Page Dataset) - KNN Baseline")
plt.tight_layout()
plt.show()

### **Performance Comparison of Traditional ML Models (Web Page Dataset)**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

metrics_web = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "KNN"],
    "Accuracy": [
        accuracy_score(y_web_test, y_pred_lr),
        accuracy_score(y_web_test, y_pred_rf),
        accuracy_score(y_web_test, y_pred_knn)
    ],
    "Recall (Phishing=1)": [
        float(classification_report(y_web_test, y_pred_lr, output_dict=True)['1']['recall']),
        float(classification_report(y_web_test, y_pred_rf, output_dict=True)['1']['recall']),
        float(classification_report(y_web_test, y_pred_knn, output_dict=True)['1']['recall'])
    ],
    "F1-score (Phishing=1)": [
        float(classification_report(y_web_test, y_pred_lr, output_dict=True)['1']['f1-score']),
        float(classification_report(y_web_test, y_pred_rf, output_dict=True)['1']['f1-score']),
        float(classification_report(y_web_test, y_pred_knn, output_dict=True)['1']['f1-score'])
    ],
    "ROC-AUC": [
        roc_auc_score(y_web_test, y_prob_lr),
        roc_auc_score(y_web_test, y_prob_rf),
        roc_auc_score(y_web_test, y_prob_knn)
    ]
}).set_index("Model")

metrics_web_percent = (metrics_web * 100).round(1)
display(metrics_web_percent.style.format("{:.1f}%"))

plt.figure(figsize=(9, 3.5))
sns.heatmap(metrics_web_percent, annot=True, fmt=".1f", cmap="Greens", linewidths=0.5)
plt.title("Traditional ML Models Comparison (Web Page Dataset) - Metrics (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

vals = metrics_web_percent.values
models = metrics_web_percent.index.tolist()
cols = metrics_web_percent.columns.tolist()

x = np.arange(len(models))
width = 0.2

plt.figure(figsize=(10, 4.5))
colors_green = ["#1b5e20", "#2e7d32", "#43a047", "#66bb6a"]
for j, col in enumerate(cols):
    plt.bar(x + j * width - width * 1.5, vals[:, j], width, label=col, color=colors_green[j])

plt.ylim(0, 105)
plt.ylabel("Percentage (%)")
plt.title("Traditional ML Models Comparison (Web Page Dataset)")
plt.xticks(x, models)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

---

### 2.2 Baseline Models - URL Features Dataset

In this section, the same three baseline classifiers are applied to the URL Features dataset
(extracted in Week 1) to establish performance benchmarks on both datasets.

**Logistic Regression - URL Features Dataset**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Train model on URL features
url_lr = LogisticRegression(max_iter=1000)
url_lr.fit(X_url_train, y_url_train)

# Predict
y_pred_url_lr = url_lr.predict(X_url_test)
y_prob_url_lr = url_lr.predict_proba(X_url_test)[:, 1]

# Evaluate
print(" Logistic Regression - Baseline Results (URL Features Dataset)")
print("Accuracy:", accuracy_score(y_url_test, y_pred_url_lr))
print("ROC-AUC:", roc_auc_score(y_url_test, y_prob_url_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_url_test, y_pred_url_lr))
print("\nClassification Report:\n", classification_report(y_url_test, y_pred_url_lr))

**Confusion Matrix (URL Features Dataset) - Logistic Regression**

In [ ]:
cm_url_lr = confusion_matrix(y_url_test, y_pred_url_lr)

plt.figure(figsize=(6,4))
sns.heatmap(cm_url_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Legitimate (0)", "Phishing (1)"],
            yticklabels=["Legitimate (0)", "Phishing (1)"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (URL Features Dataset) - Logistic Regression")
plt.tight_layout()
plt.show()

**Random Forest - URL Features Dataset**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train
url_rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
url_rf.fit(X_url_train, y_url_train)

# Predict
y_pred_url_rf = url_rf.predict(X_url_test)
y_prob_url_rf = url_rf.predict_proba(X_url_test)[:, 1]

# Evaluate
print(" Random Forest - Baseline Results (URL Features Dataset)")
print("Accuracy:", accuracy_score(y_url_test, y_pred_url_rf))
print("ROC-AUC:", roc_auc_score(y_url_test, y_prob_url_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_url_test, y_pred_url_rf))
print("\nClassification Report:\n", classification_report(y_url_test, y_pred_url_rf))

**Confusion Matrix (URL Features Dataset) - Random Forest**

In [ ]:
cm_url_rf = confusion_matrix(y_url_test, y_pred_url_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm_url_rf, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (URL Features Dataset) - Random Forest")
plt.tight_layout()
plt.show()

**KNN - URL Features Dataset**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Build KNN pipeline
url_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

# Train on FULL training set (KNN is fast, no need to sample)
url_knn.fit(X_url_train, y_url_train)

# Predict on test set
y_pred_url_knn = url_knn.predict(X_url_test)
y_prob_url_knn = url_knn.predict_proba(X_url_test)[:, 1]

# Evaluate
print(" KNN - Baseline Results (URL Features Dataset)")
print("Accuracy:", accuracy_score(y_url_test, y_pred_url_knn))
print("ROC-AUC:", roc_auc_score(y_url_test, y_prob_url_knn))
print("\nConfusion Matrix:\n", confusion_matrix(y_url_test, y_pred_url_knn))
print("\nClassification Report:\n", classification_report(y_url_test, y_pred_url_knn))

**Confusion Matrix (URL Features Dataset) - KNN**

In [ ]:
cm_url_knn = confusion_matrix(y_url_test, y_pred_url_knn)

plt.figure(figsize=(6,4))
sns.heatmap(cm_url_knn, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (URL Features Dataset) - KNN")
plt.tight_layout()
plt.show()

### **Performance Comparison of Traditional ML Models (URL Features Dataset)**

In [ ]:
metrics_url = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "KNN"],
    "Accuracy": [
        accuracy_score(y_url_test, y_pred_url_lr),
        accuracy_score(y_url_test, y_pred_url_rf),
        accuracy_score(y_url_test, y_pred_url_knn)
    ],
    "Recall (Phishing=1)": [
        float(classification_report(y_url_test, y_pred_url_lr, output_dict=True)['1']['recall']),
        float(classification_report(y_url_test, y_pred_url_rf, output_dict=True)['1']['recall']),
        float(classification_report(y_url_test, y_pred_url_knn, output_dict=True)['1']['recall'])
    ],
    "F1-score (Phishing=1)": [
        float(classification_report(y_url_test, y_pred_url_lr, output_dict=True)['1']['f1-score']),
        float(classification_report(y_url_test, y_pred_url_rf, output_dict=True)['1']['f1-score']),
        float(classification_report(y_url_test, y_pred_url_knn, output_dict=True)['1']['f1-score'])
    ],
    "ROC-AUC": [
        roc_auc_score(y_url_test, y_prob_url_lr),
        roc_auc_score(y_url_test, y_prob_url_rf),
        roc_auc_score(y_url_test, y_prob_url_knn)
    ]
}).set_index("Model")

metrics_url_percent = (metrics_url * 100).round(1)
display(metrics_url_percent.style.format("{:.1f}%"))

plt.figure(figsize=(9, 3.5))
sns.heatmap(metrics_url_percent, annot=True, fmt=".1f", cmap="Blues", linewidths=0.5)
plt.title("Traditional ML Models Comparison (URL Features Dataset) - Metrics (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

vals_url = metrics_url_percent.values
models_url = metrics_url_percent.index.tolist()
cols_url = metrics_url_percent.columns.tolist()

x = np.arange(len(models_url))
width = 0.2

plt.figure(figsize=(10, 4.5))
colors_blue = ["#0d47a1", "#1565c0", "#1e88e5", "#42a5f5"]
for j, col in enumerate(cols_url):
    plt.bar(x + j * width - width * 1.5, vals_url[:, j], width, label=col, color=colors_blue[j])

plt.ylim(0, 105)
plt.ylabel("Percentage (%)")
plt.title("Traditional ML Models Comparison (URL Features Dataset)")
plt.xticks(x, models_url)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Week 3 - Baseline Model Performance Evaluation

### 3.1 Web Page Dataset - Baseline Results

#### 3.1.1 Logistic Regression

The Logistic Regression model was implemented as the initial baseline classifier using the preprocessed Web Page phishing dataset. The dataset was split into 80% for training and 20% for testing using stratified sampling to preserve class distribution.

The model achieved an overall accuracy of **76.8%** on the test set, with a ROC-AUC of **83.5%**.

The confusion matrix shows that the model correctly classified:

* **535 legitimate websites (True Negatives)**
* **2827 phishing websites (True Positives)**

However:

* **669 legitimate websites** were misclassified as phishing (False Positives).
* **348 phishing websites** were misclassified as legitimate (False Negatives).

The classification report indicates a **high recall score of 0.89 for phishing websites (class 1)**, showing strong ability to detect malicious websites. However, the recall for legitimate websites (0.44) is significantly lower, indicating that the model tends to classify legitimate websites as phishing.

From a cybersecurity perspective, prioritising phishing detection is desirable, as failing to detect phishing attacks poses greater risk than raising false alarms. Nevertheless, the number of false positives highlights the need for further optimisation.

#### 3.1.2 Random Forest

To further evaluate traditional machine learning approaches, a Random Forest classifier was implemented using the same training and testing splits.

The Random Forest model achieved an accuracy of **74.4%** and a ROC-AUC of **81.1%**, slightly lower than Logistic Regression.

Although Random Forest maintained reasonable phishing detection performance (recall = 0.84 for class 1), it did not outperform Logistic Regression in overall accuracy or class balance.

This suggests that, for the Web Page dataset, Random Forest does not provide significant improvement over the simpler linear model.

#### 3.1.3 K-Nearest Neighbours (KNN)

A K-Nearest Neighbours (KNN) classifier with k=5 was trained and evaluated under identical conditions. KNN classifies each sample based on the majority label among its five closest neighbours in the feature space.

The KNN model achieved an accuracy of **77.0%** and a ROC-AUC of **82.8%** on the Web Page dataset.

The classification report shows that KNN achieved:

* **Recall of 0.85 for phishing websites (class 1)**
* F1-score of 0.84 for phishing detection

KNN performed comparably to Logistic Regression in terms of overall accuracy, with slightly lower phishing recall (0.85 vs 0.89). This suggests that distance-based classification captures similar patterns to the linear model on this dataset.

---

### 3.2 URL Features Dataset - Baseline Results

The same three classifiers were applied to the URL Features dataset, where 16 lexical features were extracted from raw URLs during preprocessing (Week 1). This enables a direct comparison of model performance across both datasets.

#### 3.2.1 Logistic Regression

Logistic Regression achieved an accuracy of **83.4%** on the URL Features dataset. While this appears higher than its performance on the Web Page dataset (76.8%), the model exhibited a critically low **recall of 0.34 for phishing URLs (class 1)**, meaning it failed to detect approximately **66% of phishing URLs** (15,137 out of 22,860). The ROC-AUC score was **79.5%**.

This poor phishing recall indicates that the extracted lexical features alone are insufficient for Logistic Regression to distinguish between legitimate and phishing URLs effectively.

#### 3.2.2 Random Forest

Random Forest achieved the strongest performance on the URL Features dataset, with an accuracy of **90.9%** and a ROC-AUC of **94.4%**. The recall for phishing URLs improved significantly to **0.74**, correctly identifying 16,834 out of 22,860 phishing URLs.

This shows that Random Forest is better suited for capturing non-linear patterns in URL lexical features compared to the other traditional models.

#### 3.2.3 K-Nearest Neighbours (KNN)

KNN achieved an accuracy of **90.3%** and a ROC-AUC of **91.6%** on the URL Features dataset, with a phishing recall of **0.72**. This performance is close to Random Forest, confirming that the URL lexical features contain strong neighbourhood-based patterns that distance-based classifiers can exploit effectively.

Unlike the Web Page dataset where KNN performed similarly to LR, on the URL Features dataset KNN significantly outperformed Logistic Regression (90.3% vs 83.4% accuracy), suggesting that the URL features contain non-linear relationships that KNN captures well.

---

### 3.3 Comparative Analysis

**Web Page Dataset:**

| Model               | Accuracy | Recall (Phishing) | F1-score (Phishing) | ROC-AUC |
| ------------------- | -------- | ------------------ | -------------------- | ------- |
| Logistic Regression | 76.8%    | 89.0%              | 84.8%                | 83.5%   |
| Random Forest       | 74.4%    | 84.0%              | 82.6%                | 81.1%   |
| KNN                 | 77.0%    | 85.5%              | 84.4%                | 82.8%   |

**URL Features Dataset:**

| Model               | Accuracy | Recall (Phishing) | F1-score (Phishing) | ROC-AUC |
| ------------------- | -------- | ------------------ | -------------------- | ------- |
| Logistic Regression | 83.4%    | 33.8%              | 47.9%                | 79.5%   |
| Random Forest       | 90.9%    | 73.6%              | 78.5%                | 94.4%   |
| KNN                 | 90.3%    | 71.8%              | 76.9%                | 91.6%   |

**Key findings:**

On the Web Page dataset, **Logistic Regression and KNN showed the strongest overall performance**, achieving the highest accuracy (76.8% and 77.0% respectively) and the best recall for phishing detection (89.0% and 85.5%). This dataset contains pre-engineered numerical features that capture HTML and structural properties of web pages, which both linear and distance-based classifiers can use effectively.

On the URL Features dataset, **Random Forest achieved the best results** with 90.9% accuracy and 94.4% ROC-AUC, closely followed by KNN (90.3% accuracy, 91.6% ROC-AUC). However, even the best-performing model missed approximately **26% of phishing URLs**, and Logistic Regression missed **66%** of phishing URLs. This highlights a fundamental limitation: **manually extracted lexical features from URLs do not capture sufficient information** to reliably detect phishing, as many phishing URLs are designed to closely mimic legitimate URLs in structure and appearance.

The inclusion of ROC-AUC scores provides a broader evaluation metric that accounts for the trade-off between true positive rate and false positive rate, as required by the project plan.

These results establish clear performance benchmarks and strongly justify the need for the transfer learning approach (DistilBERT) in the next phase, which can learn deeper semantic patterns directly from raw URL text rather than relying on manually engineered features.

# Week 4 - Transfer Learning (Pre-training Phase)

## 4.1 DistilBERT Pre-training on URL Dataset

In this phase, a pre-trained DistilBERT model is fine-tuned on the Phishing Site URLs dataset to learn deep semantic patterns directly from raw URL text. This approach addresses the limitations identified in the baseline evaluation (Week 3), where manually extracted lexical features proved insufficient for reliable phishing detection.

DistilBERT was selected as it retains 97% of BERT's language understanding while being 60% faster and smaller, making it well-suited for Google Colab's computational constraints.

### 4.1.1 Install Required Libraries

In [ ]:
!pip install transformers accelerate -q

In [ ]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings('ignore')

# Set device (GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### 4.1.3 Prepare URL Data for DistilBERT

The raw URL text from the cleaned dataset is used directly as input to DistilBERT. The full cleaned URL dataset (507,196 samples) is used for fine-tuning.

In [ ]:
# Use the cleaned URL dataset from Week 1.
# IMPORTANT: We reuse the EXACT split positions computed in Cell 13
# (url_train_pos / url_test_pos) so that DistilBERT's test set is the same
# URLs whose extracted features were tested against baselines.  Without this,
# the comparison in Section 4.1.11 would be on different test populations.
from sklearn.model_selection import train_test_split

# Map the split positions to the raw URL strings
df_url_clean_seq = df_url_clean.reset_index(drop=True)  # ensure 0..N-1 indexing
url_text_array  = df_url_clean_seq['URL'].values
url_label_array = df_url_clean_seq['Label_encoded'].values

X_train_text_full = url_text_array[url_train_pos]
y_train_labels_full = url_label_array[url_train_pos]
X_test_text  = url_text_array[url_test_pos]
y_test_labels = url_label_array[url_test_pos]

# Hold out 10% of the training portion as a validation set
X_train_text, X_val_text, y_train_labels, y_val_labels = train_test_split(
    X_train_text_full, y_train_labels_full,
    test_size=0.1,
    random_state=42,
    stratify=y_train_labels_full
)

print(f"Training set:   {len(X_train_text)} samples")
print(f"Validation set: {len(X_val_text)} samples")
print(f"Test set:       {len(X_test_text)} samples")
print(f"\nTraining label distribution:")
print(f"  Legitimate (0): {sum(y_train_labels == 0)} ({sum(y_train_labels == 0)/len(y_train_labels)*100:.1f}%)")
print(f"  Phishing (1):   {sum(y_train_labels == 1)} ({sum(y_train_labels == 1)/len(y_train_labels)*100:.1f}%)")
print("\n DistilBERT test set is the SAME URLs as the baseline test set (Cell 13).")


### 4.1.4 Tokenization

DistilBERT uses WordPiece tokenization to convert raw URL text into token IDs that the model can process.

In [ ]:
# Load DistilBERT tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Tokenize a sample URL to demonstrate the process
sample_url = X_train_text[0]
sample_tokens = tokenizer(sample_url, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

print("Sample URL:", sample_url)
print("\nToken IDs:", sample_tokens['input_ids'][0][:20], "...")
print("Attention Mask:", sample_tokens['attention_mask'][0][:20], "...")
print(f"\nTotal tokens: {sample_tokens['input_ids'].shape[1]}")

### 4.1.5 Create PyTorch Dataset

In [ ]:
class URLDataset(Dataset):
    def __init__(self, urls, labels, tokenizer, max_length=128):
        self.urls = urls
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = str(self.urls[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            url,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = URLDataset(X_train_text, y_train_labels, tokenizer)
val_dataset = URLDataset(X_val_text, y_val_labels, tokenizer)
test_dataset = URLDataset(X_test_text, y_test_labels, tokenizer)

# Create DataLoaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

### 4.1.6 Load and Configure DistilBERT Model

In [ ]:
# Load pre-trained DistilBERT with a classification head (2 classes)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.to(device)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

# Mixed precision scaler for faster training
scaler = GradScaler()

print(f"Model loaded on {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

### 4.1.7 Training Loop

The model is trained for 3 epochs with mixed-precision training for efficiency on Colab GPU.

In [ ]:
def train_epoch(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {correct/total:.4f}")

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, accuracy, all_preds, all_labels, all_probs

In [ ]:
# Training
EPOCHS = 3
train_losses = []
val_losses = []
train_accs = []
val_accs = []

print("Starting DistilBERT Training on URL Dataset")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scaler, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # Validate
    val_loss, val_acc, _, _, _ = evaluate(model, val_loader, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

print(" Training Complete!")

### 4.1.8 Training Progress Visualisation

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss plot
ax1.plot(range(1, EPOCHS+1), train_losses, 'b-o', label='Train Loss')
ax1.plot(range(1, EPOCHS+1), val_losses, 'r-o', label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(range(1, EPOCHS+1), train_accs, 'b-o', label='Train Accuracy')
ax2.plot(range(1, EPOCHS+1), val_accs, 'r-o', label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.1.9 Evaluate on Test Set

In [ ]:
# Evaluate on test set
test_loss, test_acc, test_preds, test_labels, test_probs = evaluate(model, test_loader, device)

print(" DistilBERT - Test Results (URL Dataset)")
print(f"Accuracy: {test_acc:.4f}")
print(f"ROC-AUC: {roc_auc_score(test_labels, test_probs):.4f}")
print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=['Legitimate (0)', 'Phishing (1)']))

### 4.1.10 Confusion Matrix

In [ ]:
cm_bert = confusion_matrix(test_labels, test_preds)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Legitimate (0)', 'Phishing (1)'],
            yticklabels=['Legitimate (0)', 'Phishing (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (URL Dataset) - DistilBERT')
plt.tight_layout()
plt.show()

### 4.1.11 Comparison with Baseline Models

This comparison shows the improvement gained by using DistilBERT (transfer learning) over the traditional ML baselines on the URL dataset.

In [ ]:
# Get DistilBERT metrics
bert_report = classification_report(test_labels, test_preds, output_dict=True)

# Find the correct key for phishing class in bert_report
bert_keys = list(bert_report.keys())
print("DistilBERT report keys:", bert_keys)

# Get phishing recall and f1 from DistilBERT
bert_phishing_key = [k for k in bert_keys if '1' in str(k)][0]
bert_recall = float(bert_report[bert_phishing_key]['recall'])
bert_f1 = float(bert_report[bert_phishing_key]['f1-score'])

# Comparison table
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "KNN", "DistilBERT"],
    "Accuracy": [
        accuracy_score(y_url_test, y_pred_url_lr),
        accuracy_score(y_url_test, y_pred_url_rf),
        accuracy_score(y_url_test, y_pred_url_knn),
        test_acc
    ],
    "Recall (Phishing=1)": [
        float(classification_report(y_url_test, y_pred_url_lr, output_dict=True)['1']['recall']),
        float(classification_report(y_url_test, y_pred_url_rf, output_dict=True)['1']['recall']),
        float(classification_report(y_url_test, y_pred_url_knn, output_dict=True)['1']['recall']),
        bert_recall
    ],
    "F1-score (Phishing=1)": [
        float(classification_report(y_url_test, y_pred_url_lr, output_dict=True)['1']['f1-score']),
        float(classification_report(y_url_test, y_pred_url_rf, output_dict=True)['1']['f1-score']),
        float(classification_report(y_url_test, y_pred_url_knn, output_dict=True)['1']['f1-score']),
        bert_f1
    ],
    "ROC-AUC": [
        roc_auc_score(y_url_test, y_prob_url_lr),
        roc_auc_score(y_url_test, y_prob_url_rf),
        roc_auc_score(y_url_test, y_prob_url_knn),
        roc_auc_score(test_labels, test_probs)
    ]
}).set_index("Model")

comparison_percent = (comparison * 100).round(1)
display(comparison_percent.style.format("{:.1f}%"))

plt.figure(figsize=(10, 4))
sns.heatmap(comparison_percent, annot=True, fmt=".1f", cmap="RdYlGn", linewidths=0.5)
plt.title("DistilBERT vs Traditional ML Baselines (URL Dataset) - Metrics (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 4.1.12 Save Pre-trained Model

The fine-tuned DistilBERT model is saved for use in the transfer learning phase (Week 5), where its learned representations will be applied to the Web Page dataset.

In [ ]:
# Save the fine-tuned model
model.save_pretrained('/content/distilbert_url_pretrained')
tokenizer.save_pretrained('/content/distilbert_url_pretrained')

print(" Pre-trained DistilBERT model saved successfully!")
print("   Location: /content/distilbert_url_pretrained")
print("   This model will be used in Week 5 for transfer learning to the Web Page dataset.")

### 4.1.13 Week 4 Summary

- DistilBERT was successfully fine-tuned on the Phishing Site URLs dataset using raw URL text as input.
- The model was trained for 3 epochs using mixed-precision training on Google Colab GPU.
- Performance was evaluated using accuracy, precision, recall, F1-score, and ROC-AUC in the project plan.
- Results were compared against the traditional ML baselines (Logistic Regression, Random Forest, KNN) from Week 2.
- The fine-tuned model was saved for the transfer learning phase in Week 5, where learned URL representations will be applied to the Web Page phishing dataset.

This shows that deep transfer learning using DistilBERT can capture semantic patterns in URL text that are not accessible through manually engineered lexical features, addressing a key limitation identified in the baseline evaluation.

# Week 5 - Transfer Learning (Fine-tuning Phase)

This phase implements cross-dataset transfer learning in the project plan: "pre-trained on one dataset and fine-tuned on the other." Three distinct approaches are explored to address the challenge that the URL dataset contains raw text while the Web Page dataset contains only numerical features.

- **Approach 1:** DistilBERT Embeddings + Web Page Features (embedding concatenation)
- **Approach 2:** MLP Transfer Learning (shared numerical feature space)
- **Approach 3:** Pseudo-Text Generation + DistilBERT Fine-tuning (feature-to-text conversion)

All approaches are evaluated on the **same** Web Page test set (from the Week 1 split) for fair comparison.

---

## 5.1 Approach 1: DistilBERT Embeddings + Web Page Features

In the previous phase (Week 4), DistilBERT was fine-tuned on the Phishing Site URLs dataset, learning deep semantic representations from raw URL text. In this approach, the pre-trained DistilBERT model is used as a feature extractor: URL embeddings generated by the model are combined with the numerical features from the Web Page Phishing dataset to create an combined feature set. A new classifier is then trained on these combined features, achieving cross-dataset transfer learning in the project plan.

This approach addresses the key challenge that the two datasets have fundamentally different formats (text vs. numerical), enabling knowledge transfer between them through learned representations.

### 5.1.1 Load the Pre-trained DistilBERT Model

The fine-tuned DistilBERT model saved in Week 4 is loaded to extract URL embeddings.

In [ ]:
# Load the fine-tuned DistilBERT model from Week 4
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizer and fine-tuned model
tokenizer = DistilBertTokenizer.from_pretrained('/content/distilbert_url_pretrained')
finetuned_model = DistilBertForSequenceClassification.from_pretrained('/content/distilbert_url_pretrained')

# Extract the base DistilBERT (without classification head) for embeddings
embedding_model = finetuned_model.distilbert
embedding_model.to(device)
embedding_model.eval()

print(" Pre-trained DistilBERT loaded successfully!")
print(f"   Model parameters: {sum(p.numel() for p in embedding_model.parameters()):,}")

### 5.1.2 Generate URL Embeddings for Web Page Dataset

Since the Web Page dataset does not contain raw URL text, we generate embeddings from a representative sample of URLs from the URL dataset. These embeddings capture the semantic patterns DistilBERT learned about phishing vs. legitimate URLs.

In [ ]:
print("Web Page Dataset:")
print(f"  Total samples: {len(df_web_clean)}")
print(f"  Label distribution:")
print(f"    Legitimate (0): {sum(df_web_clean['phishing'] == 0)} ({sum(df_web_clean['phishing'] == 0)/len(df_web_clean)*100:.1f}%)")
print(f"    Phishing (1): {sum(df_web_clean['phishing'] == 1)} ({sum(df_web_clean['phishing'] == 1)/len(df_web_clean)*100:.1f}%)")

n_legit = sum(df_web_clean['phishing'] == 0)
n_phish = sum(df_web_clean['phishing'] == 1)

url_legit = df_url_clean[df_url_clean['Label_encoded'] == 0].sample(n=min(n_legit, sum(df_url_clean['Label_encoded'] == 0)), random_state=42)
url_phish = df_url_clean[df_url_clean['Label_encoded'] == 1].sample(n=min(n_phish, sum(df_url_clean['Label_encoded'] == 1)), random_state=42)

url_sample = pd.concat([url_legit, url_phish]).reset_index(drop=True)
print(f"\nSampled {len(url_sample)} URLs for embedding extraction")
print(f"  Legitimate: {len(url_legit)}, Phishing: {len(url_phish)}")

### 5.1.3 Extract [CLS] Token Embeddings

The [CLS] token embedding from DistilBERT represents the overall semantic meaning of each URL. These 768-dimensional vectors capture patterns that distinguish phishing from legitimate URLs.

In [ ]:
def extract_embeddings(model, tokenizer, urls, device, batch_size=64):
    """Extract [CLS] token embeddings from DistilBERT for a list of URLs."""
    model.eval()
    all_embeddings = []

    for i in range(0, len(urls), batch_size):
        batch_urls = urls[i:i+batch_size]

        encoding = tokenizer(
            batch_urls.tolist(),
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].to(device)
        attention_mask = encoding['attention_mask'].to(device)

        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            cls_embeddings = outputs.last_hidden_state[:, 0, :]

        all_embeddings.append(cls_embeddings.cpu().numpy())

        if (i // batch_size + 1) % 50 == 0:
            print(f"  Processed {i + len(batch_urls)}/{len(urls)} URLs...")

    return np.vstack(all_embeddings)

print("Extracting DistilBERT embeddings from sampled URLs...")
print("This may take a few minutes...\n")

url_texts = url_sample['URL'].values
url_embeddings = extract_embeddings(embedding_model, tokenizer, url_texts, device)

print(f"\n Embedding extraction complete!")
print(f"   Shape: {url_embeddings.shape}")
print(f"   Each URL is represented as a {url_embeddings.shape[1]}-dimensional vector")

### 5.1.4 Combine Embeddings with Web Page Features

The DistilBERT embeddings (768 dimensions) are concatenated with the Web Page numerical features to create an combined feature set. This is the core of the cross-dataset transfer learning approach.

In [ ]:
# Get Web Page features
webpage_feature_cols = [col for col in df_web_clean.columns if col != 'phishing']
X_webpage = df_web_clean[webpage_feature_cols].values
y_webpage = df_web_clean['phishing'].values

print(f"Web Page features shape: {X_webpage.shape}")
print(f"URL embeddings shape: {url_embeddings.shape}")

# Ensure same number of samples
min_samples = min(len(X_webpage), len(url_embeddings))
X_webpage_trimmed = X_webpage[:min_samples]
url_embeddings_trimmed = url_embeddings[:min_samples]
y_combined = y_webpage[:min_samples]

# Concatenate: [Web Page features | DistilBERT embeddings]
X_combined = np.hstack([X_webpage_trimmed, url_embeddings_trimmed])

print(f"\n Combined feature set:")
print(f"   Web Page features: {X_webpage_trimmed.shape[1]} dimensions")
print(f"   DistilBERT embeddings: {url_embeddings_trimmed.shape[1]} dimensions")
print(f"   Combined: {X_combined.shape[1]} dimensions")
print(f"   Total samples: {X_combined.shape[0]}")

### 5.1.5 Train-Test Split

The combined feature set is split into training and testing sets using stratified sampling to maintain class distribution.

In [ ]:
# Split combined features
X_train_combined, X_test_combined, y_train_combined, y_test_combined = train_test_split(
    X_combined, y_combined,
    test_size=0.2,
    random_state=42,
    stratify=y_combined
)

# Scale features
scaler_combined = StandardScaler()
X_train_scaled = scaler_combined.fit_transform(X_train_combined)
X_test_scaled = scaler_combined.transform(X_test_combined)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set: {X_test_scaled.shape[0]} samples")
print(f"Feature dimensions: {X_train_scaled.shape[1]}")
print(f"\nLabel distribution (train):")
print(f"  Legitimate (0): {sum(y_train_combined == 0)} ({sum(y_train_combined == 0)/len(y_train_combined)*100:.1f}%)")
print(f"  Phishing (1): {sum(y_train_combined == 1)} ({sum(y_train_combined == 1)/len(y_train_combined)*100:.1f}%)")

### 5.1.6 Train Classifiers on Combined Features

Multiple classifiers are trained on the combined feature set (Web Page + DistilBERT embeddings) to evaluate the impact of transfer learning.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

print("Training classifiers on combined features")
print("(Web Page features + DistilBERT URL embeddings)")

# 1. Logistic Regression
print("\n[1/3] Training Logistic Regression...")
lr_combined = LogisticRegression(max_iter=1000, random_state=42)
lr_combined.fit(X_train_scaled, y_train_combined)
y_pred_lr_comb = lr_combined.predict(X_test_scaled)
y_prob_lr_comb = lr_combined.predict_proba(X_test_scaled)[:, 1]
print(f"  Accuracy: {accuracy_score(y_test_combined, y_pred_lr_comb):.4f}")

# 2. Random Forest
print("\n[2/3] Training Random Forest...")
rf_combined = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_combined.fit(X_train_scaled, y_train_combined)
y_pred_rf_comb = rf_combined.predict(X_test_scaled)
y_prob_rf_comb = rf_combined.predict_proba(X_test_scaled)[:, 1]
print(f"  Accuracy: {accuracy_score(y_test_combined, y_pred_rf_comb):.4f}")

# 3. KNN
print("\n[3/3] Training KNN...")
knn_combined = KNeighborsClassifier(n_neighbors=5)
knn_combined.fit(X_train_scaled, y_train_combined)
y_pred_knn_comb = knn_combined.predict(X_test_scaled)
y_prob_knn_comb = knn_combined.predict_proba(X_test_scaled)[:, 1]
print(f"  Accuracy: {accuracy_score(y_test_combined, y_pred_knn_comb):.4f}")

print(" All classifiers trained successfully!")

### 5.1.7 Evaluation Results

Detailed evaluation of each classifier on the combined features, including all five TOR-required metrics.

In [ ]:
models_comb = {
    'Logistic Regression': (y_pred_lr_comb, y_prob_lr_comb),
    'Random Forest': (y_pred_rf_comb, y_prob_rf_comb),
    'KNN': (y_pred_knn_comb, y_prob_knn_comb)
}

print("TRANSFER LEARNING RESULTS - Combined Features")
print("(Web Page numerical features + DistilBERT URL embeddings)")

for name, (preds, probs) in models_comb.items():
    print(f"\n{'─' * 50}")
    print(f"  {name}")
    print(f"{'─' * 50}")
    print(f"  Accuracy:  {accuracy_score(y_test_combined, preds):.4f}")
    print(f"  ROC-AUC:   {roc_auc_score(y_test_combined, probs):.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test_combined, preds, target_names=['Legitimate (0)', 'Phishing (1)']))

### 5.1.8 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, (preds, _)) in enumerate(models_comb.items()):
    cm = confusion_matrix(y_test_combined, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[idx],
                xticklabels=['Legitimate (0)', 'Phishing (1)'],
                yticklabels=['Legitimate (0)', 'Phishing (1)'])
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_title(f'{name}\n(Combined Features)')

plt.suptitle('Confusion Matrices - Transfer Learning (Week 5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1.9 Comparison: Transfer Learning vs. Baselines

This comparison shows the impact of adding DistilBERT URL embeddings to the Web Page features. The Week 2 baselines used only the Web Page numerical features, while the Week 5 models use the combined features (Web Page + DistilBERT embeddings).

In [ ]:
# Build comparison table: Week 2 Baselines vs Week 5 Transfer Learning
baseline_lr_report = classification_report(y_web_test, y_pred_lr, output_dict=True)
baseline_rf_report = classification_report(y_web_test, y_pred_rf, output_dict=True)
baseline_knn_report = classification_report(y_web_test, y_pred_knn, output_dict=True)

tl_lr_report = classification_report(y_test_combined, y_pred_lr_comb, output_dict=True)
tl_rf_report = classification_report(y_test_combined, y_pred_rf_comb, output_dict=True)
tl_knn_report = classification_report(y_test_combined, y_pred_knn_comb, output_dict=True)

comparison_full = pd.DataFrame({
    "Model": [
        "LR (Web Page only)", "RF (Web Page only)", "KNN (Web Page only)",
        "LR + DistilBERT", "RF + DistilBERT", "KNN + DistilBERT"
    ],
    "Accuracy": [
        accuracy_score(y_web_test, y_pred_lr),
        accuracy_score(y_web_test, y_pred_rf),
        accuracy_score(y_web_test, y_pred_knn),
        accuracy_score(y_test_combined, y_pred_lr_comb),
        accuracy_score(y_test_combined, y_pred_rf_comb),
        accuracy_score(y_test_combined, y_pred_knn_comb)
    ],
    "Recall (Phishing)": [
        float(baseline_lr_report['1']['recall']),
        float(baseline_rf_report['1']['recall']),
        float(baseline_knn_report['1']['recall']),
        float(tl_lr_report['1']['recall']),
        float(tl_rf_report['1']['recall']),
        float(tl_knn_report['1']['recall'])
    ],
    "F1-score (Phishing)": [
        float(baseline_lr_report['1']['f1-score']),
        float(baseline_rf_report['1']['f1-score']),
        float(baseline_knn_report['1']['f1-score']),
        float(tl_lr_report['1']['f1-score']),
        float(tl_rf_report['1']['f1-score']),
        float(tl_knn_report['1']['f1-score'])
    ],
    "ROC-AUC": [
        roc_auc_score(y_web_test, y_prob_lr),
        roc_auc_score(y_web_test, y_prob_rf),
        roc_auc_score(y_web_test, y_prob_knn),
        roc_auc_score(y_test_combined, y_prob_lr_comb),
        roc_auc_score(y_test_combined, y_prob_rf_comb),
        roc_auc_score(y_test_combined, y_prob_knn_comb)
    ]
}).set_index("Model")

comparison_pct = (comparison_full * 100).round(1)
display(comparison_pct.style.format("{:.1f}%"))

In [ ]:
# Visual comparison
fig, axes = plt.subplots(1, 4, figsize=(20, 6))

metrics = ['Accuracy', 'Recall (Phishing)', 'F1-score (Phishing)', 'ROC-AUC']

for idx, metric in enumerate(metrics):
    baseline_vals = comparison_pct.iloc[:3][metric].values
    transfer_vals = comparison_pct.iloc[3:][metric].values

    x = np.arange(3)
    width = 0.35

    axes[idx].bar(x - width/2, baseline_vals, width, label='Baseline (Web Page only)',
                  color='#85C1E9', edgecolor='#2980B9')
    axes[idx].bar(x + width/2, transfer_vals, width, label='+ DistilBERT Embeddings',
                  color='#7D3C98', edgecolor='#512E5F')

    axes[idx].set_ylabel(f'{metric} (%)')
    axes[idx].set_title(metric)
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(['LR', 'RF', 'KNN'], rotation=0)
    axes[idx].legend(fontsize=8)
    axes[idx].set_ylim(0, 105)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('Baseline vs Transfer Learning - Web Page Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1.10 Feature Importance Analysis

Examining which features contribute most to the transfer learning classifier, showing the relative importance of Web Page features vs. DistilBERT embeddings.

In [ ]:
importances = rf_combined.feature_importances_

n_webpage_features = X_webpage_trimmed.shape[1]
n_embedding_features = url_embeddings_trimmed.shape[1]

webpage_importance = importances[:n_webpage_features].sum()
embedding_importance = importances[n_webpage_features:].sum()

print(f"Feature importance breakdown:")
print(f"  Web Page features ({n_webpage_features} dims): {webpage_importance:.4f} ({webpage_importance/(webpage_importance+embedding_importance)*100:.1f}%)")
print(f"  DistilBERT embeddings ({n_embedding_features} dims): {embedding_importance:.4f} ({embedding_importance/(webpage_importance+embedding_importance)*100:.1f}%)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.pie([webpage_importance, embedding_importance],
        labels=[f'Web Page Features\n({n_webpage_features} dims)',
                f'DistilBERT Embeddings\n({n_embedding_features} dims)'],
        autopct='%1.1f%%', colors=['#85C1E9', '#7D3C98'],
        startangle=90, textprops={'fontsize': 11})
ax1.set_title('Feature Importance: Web Page vs DistilBERT', fontsize=12, fontweight='bold')

top_n = 20
top_indices = np.argsort(importances)[-top_n:][::-1]
top_names = []
for i in top_indices:
    if i < n_webpage_features:
        top_names.append(webpage_feature_cols[i])
    else:
        top_names.append(f'emb_{i - n_webpage_features}')

ax2.barh(range(top_n), importances[top_indices][::-1],
         color=['#85C1E9' if i < n_webpage_features else '#7D3C98' for i in top_indices][::-1])
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(top_names[::-1], fontsize=9)
ax2.set_xlabel('Importance')
ax2.set_title(f'Top {top_n} Features (Blue=Web Page, Purple=DistilBERT)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 5.1.11 Approach 1 Summary

- The pre-trained DistilBERT model (fine-tuned on URLs in Week 4) was used as a feature extractor to generate 768-dimensional embeddings for URL samples.
- These embeddings were combined with the Web Page dataset's numerical features, creating a combined feature set that bridges both datasets.
- Three classifiers (Logistic Regression, Random Forest, KNN) were trained on the combined features.
- Results were compared against the Week 2 baselines (Web Page features only) to quantify the improvement from transfer learning.
- Feature importance analysis revealed the relative contribution of Web Page features vs. DistilBERT embeddings.

**Key Finding:** Adding DistilBERT embeddings did not improve overall performance. The URL embeddings and Web Page samples come from different datasets with no shared samples - combining them row-by-row is essentially random pairing, so the embeddings added noise rather than useful information. KNN was particularly affected due to the curse of dimensionality (ROC-AUC dropped from 82.8% to 62.4%).

This approach is included as **comparative analysis** in the project, showing the limitations of embedding concatenation for cross-dataset transfer when samples are not aligned.


---

## 5.2 Approach 2: MLP Transfer Learning (Numerical Features)

### Pre-train on URL Features → Fine-tune on Web Page Features

This approach implements transfer learning using a Multi-Layer Perceptron (MLP) neural network. Both datasets are aligned into a shared numerical feature space, enabling the MLP to be pre-trained on one dataset and fine-tuned on the other - exactly in the project plan: *"pre-trained on one dataset and fine-tuned on the other."*

**Key advantage over Approach 1:** Both datasets are numerical, so there is no text-vs-numbers mismatch.

**This is the primary proposed model** for the project, as confirmed by the supervisor.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### 5.2.1 Align Datasets to Shared Feature Space

The URL dataset has 16 features and the Web Page dataset has 19 features. After aligning the URL feature names with the Web Page naming convention (Cell 11), eight features are shared between them: `url_length`, `n_dots`, `n_hypens`, `n_underline`, `n_slash`, `n_questionmark`, `n_and`, `n_at`. We create a unified feature space of 27 features (16 + 19 − 8 shared) containing all unique features from both datasets, filling missing features with 0.

In [ ]:
# Get feature names from existing DataFrames
url_cols = list(X_url_train.columns)  # 16 URL features (renamed in Cell 11 to align with Web Page naming)
web_cols = list(X_web_train.columns)  # 19 Web Page features

print(f"URL features ({len(url_cols)}): {url_cols}")
print(f"\nWeb Page features ({len(web_cols)}): {web_cols}")

# Find shared and unique features
# After Fix #1, URL feature names align with Web Page naming where the
# underlying measurement is the same.  Set-intersection therefore identifies
# all 8 truly-shared features (url_length plus 7 character-count features),
# rather than only url_length.  This is what enables MLP transfer learning
# in Approach 2 to actually transfer knowledge between datasets.
shared = set(url_cols) & set(web_cols)
print(f"\nShared features ({len(shared)}): {sorted(shared)}")

# Create unified feature space (union of all column names)
all_features = sorted(set(url_cols + web_cols))
print(f"\nUnified feature space: {len(all_features)} features")
print(f"  {all_features}")


In [ ]:
# Align DataFrames to shared feature space (fill missing with 0)
def align_to_shared(df, all_features):
    aligned = pd.DataFrame(0.0, index=df.index, columns=all_features)
    for col in df.columns:
        if col in all_features:
            aligned[col] = df[col].values
    return aligned

# Align URL train/test (existing splits from Week 1)
X_url_train_aligned = align_to_shared(X_url_train, all_features)
X_url_test_aligned = align_to_shared(X_url_test, all_features)

# Align Web Page train/test (existing splits from Week 1)
X_web_train_aligned = align_to_shared(X_web_train, all_features)
X_web_test_aligned = align_to_shared(X_web_test, all_features)

print(f"URL train aligned: {X_url_train_aligned.shape}")
print(f"URL test aligned: {X_url_test_aligned.shape}")
print(f"Web train aligned: {X_web_train_aligned.shape}")
print(f"Web test aligned: {X_web_test_aligned.shape}")
print(f"\nUsing SAME train/test splits as baselines for fair comparison")

In [ ]:
# Scale all data using URL training data (source domain)
scaler_mlp = StandardScaler()
X_url_tr_s = scaler_mlp.fit_transform(X_url_train_aligned)
X_url_te_s = scaler_mlp.transform(X_url_test_aligned)
X_web_tr_s = scaler_mlp.transform(X_web_train_aligned)
X_web_te_s = scaler_mlp.transform(X_web_test_aligned)

print(" All datasets scaled to shared feature space")
print(f"   Feature dimensions: {X_url_tr_s.shape[1]}")

### 5.2.2 Build MLP Model


In [ ]:
class PhishingMLP(nn.Module):
    def __init__(self, input_dim):
        super(PhishingMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

input_dim = len(all_features)
print(f"MLP Architecture:")
print(f"  Input:  {input_dim} features (shared space)")
print(f"  Hidden: 128 → 64 → 32 (ReLU + Dropout 0.3)")
print(f"  Output: 1 (sigmoid → binary)")

model_mlp = PhishingMLP(input_dim).to(device)
print(f"  Parameters: {sum(p.numel() for p in model_mlp.parameters()):,}")

### 5.2.3 Training Functions


In [ ]:
def create_loaders(X_train, y_train, X_test, y_test, batch_size=256):
    train_ds = TensorDataset(
        torch.FloatTensor(X_train),
        torch.FloatTensor(y_train.values if hasattr(y_train, 'values') else y_train)
    )
    test_ds = TensorDataset(
        torch.FloatTensor(X_test),
        torch.FloatTensor(y_test.values if hasattr(y_test, 'values') else y_test)
    )
    return DataLoader(train_ds, batch_size=batch_size, shuffle=True), \
           DataLoader(test_ds, batch_size=batch_size, shuffle=False)

def train_model(model, train_loader, test_loader, epochs, lr, device):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    train_losses, test_losses, train_accs, test_accs = [], [], [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss, correct, total = 0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze()
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            preds = (outputs >= 0.5).float()
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)

        train_losses.append(epoch_loss / len(train_loader))
        train_accs.append(correct / total)

        model.eval()
        eval_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch).squeeze()
                loss = criterion(outputs, y_batch)
                eval_loss += loss.item()
                preds = (outputs >= 0.5).float()
                correct += (preds == y_batch).sum().item()
                total += len(y_batch)

        test_losses.append(eval_loss / len(test_loader))
        test_accs.append(correct / total)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | Train Acc: {train_accs[-1]:.4f} | Val Acc: {test_accs[-1]:.4f}")

    return train_losses, test_losses, train_accs, test_accs

def evaluate_model(model, X_test, y_test, device):
    model.eval()
    with torch.no_grad():
        X_t = torch.FloatTensor(X_test).to(device)
        probs = model(X_t).squeeze().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
    y_true = y_test.values if hasattr(y_test, 'values') else y_test
    return accuracy_score(y_true, preds), roc_auc_score(y_true, probs), preds, probs

### 5.2.4 Pre-train MLP on URL Dataset (Phase 1)

The MLP learns phishing detection patterns from URL numerical features.

In [ ]:
url_train_loader, url_test_loader = create_loaders(
    X_url_tr_s, y_url_train, X_url_te_s, y_url_test, batch_size=256
)

print("PHASE 1: Pre-training MLP on URL Dataset")
print(f"  Training samples: {len(X_url_tr_s)}")
print(f"  Feature space: {X_url_tr_s.shape[1]} dimensions")

model_mlp = PhishingMLP(input_dim).to(device)

pt_loss_tr, pt_loss_te, pt_acc_tr, pt_acc_te = train_model(
    model_mlp, url_train_loader, url_test_loader,
    epochs=30, lr=0.001, device=device
)

url_acc, url_roc, url_preds, url_probs_mlp = evaluate_model(
    model_mlp, X_url_te_s, y_url_test, device
)

print(f"\n Pre-training complete!")
print(f"   URL Test Accuracy: {url_acc:.4f}")
print(f"   URL Test ROC-AUC:  {url_roc:.4f}")
print(f"\nClassification Report (URL Dataset):")
print(classification_report(
    y_url_test, url_preds,
    target_names=['Legitimate (0)', 'Phishing (1)']
))

# Save pre-trained weights
torch.save(model_mlp.state_dict(), '/content/mlp_pretrained.pth')
print(" Pre-trained weights saved!")

### 5.2.5 Pre-training Visualisation


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(pt_loss_tr, 'b-', label='Train Loss')
ax1.plot(pt_loss_te, 'r-', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Pre-training Loss (URL Dataset)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(pt_acc_tr, 'b-', label='Train Accuracy')
ax2.plot(pt_acc_te, 'r-', label='Val Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Pre-training Accuracy (URL Dataset)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('MLP Pre-training on URL Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2.6 Fine-tune MLP on Web Page Dataset (Phase 2)

The **same** MLP (with pre-trained weights from URLs) is now fine-tuned on the Web Page dataset. The learning rate (0.001) is matched with the from-scratch baseline (Section 5.2.8) so that the comparison isolates the effect of pre-training rather than the effect of a different learning rate.

In [ ]:
web_train_loader, web_test_loader = create_loaders(
    X_web_tr_s, y_web_train, X_web_te_s, y_web_test, batch_size=256
)

print("PHASE 2: Fine-tuning MLP on Web Page Dataset")
print(f"  Training samples: {len(X_web_tr_s)}")
print(f"  Using pre-trained weights from Phase 1")
print(f"  Learning rate: 0.001 (matched with scratch baseline for a strict fair comparison; Fix #E)")

ft_loss_tr, ft_loss_te, ft_acc_tr, ft_acc_te = train_model(
    model_mlp, web_train_loader, web_test_loader,
    epochs=30, lr=0.001, device=device   # matched LR with scratch baseline (Fix #E)
)

web_acc_ft, web_roc_ft, web_preds_ft, web_probs_ft = evaluate_model(
    model_mlp, X_web_te_s, y_web_test, device
)

print(f"\n Fine-tuning complete!")
print(f"   Web Page Test Accuracy: {web_acc_ft:.4f}")
print(f"   Web Page Test ROC-AUC:  {web_roc_ft:.4f}")
print(f"\nClassification Report (Web Page - Fine-tuned):")
print(classification_report(
    y_web_test, web_preds_ft,
    target_names=['Legitimate (0)', 'Phishing (1)']
))

### 5.2.7 Fine-tuning Visualisation


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(ft_loss_tr, 'b-', label='Train Loss')
ax1.plot(ft_loss_te, 'r-', label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Fine-tuning Loss (Web Page Dataset)')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(ft_acc_tr, 'b-', label='Train Accuracy')
ax2.plot(ft_acc_te, 'r-', label='Val Accuracy')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Fine-tuning Accuracy (Web Page Dataset)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('MLP Fine-tuning on Web Page Dataset (Transfer Learning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2.8 Baseline - MLP from Scratch (No Transfer)

To measure the impact of transfer learning, we train an identical MLP **from scratch** on Web Page data only (no pre-training on URLs).

In [ ]:
print("BASELINE: MLP from Scratch on Web Page (No Transfer)")

model_scratch = PhishingMLP(input_dim).to(device)

sc_loss_tr, sc_loss_te, sc_acc_tr, sc_acc_te = train_model(
    model_scratch, web_train_loader, web_test_loader,
    epochs=30, lr=0.001, device=device
)

web_acc_sc, web_roc_sc, web_preds_sc, web_probs_sc = evaluate_model(
    model_scratch, X_web_te_s, y_web_test, device
)

print(f"\n Baseline complete!")
print(f"   Web Page Test Accuracy: {web_acc_sc:.4f}")
print(f"   Web Page Test ROC-AUC:  {web_roc_sc:.4f}")
print(f"\nClassification Report (Web Page - From Scratch):")
print(classification_report(
    y_web_test, web_preds_sc,
    target_names=['Legitimate (0)', 'Phishing (1)']
))

### 5.2.9 Comparison - All Models

Comparing traditional ML baselines (Week 2), MLP from scratch, and MLP with transfer learning. All evaluated on the **same** Web Page test set for fair comparison.

In [ ]:
# Baseline reports (using SAME y_web_test)
bl_lr_r = classification_report(y_web_test, y_pred_lr, output_dict=True)
bl_rf_r = classification_report(y_web_test, y_pred_rf, output_dict=True)
bl_knn_r = classification_report(y_web_test, y_pred_knn, output_dict=True)
sc_r = classification_report(y_web_test, web_preds_sc, output_dict=True)
ft_r = classification_report(y_web_test, web_preds_ft, output_dict=True)

comparison = pd.DataFrame({
    "Model": [
        "LR (Baseline)", "RF (Baseline)", "KNN (Baseline)",
        "MLP (No Transfer)", "MLP (Transfer Learning)"
    ],
    "Accuracy": [
        accuracy_score(y_web_test, y_pred_lr),
        accuracy_score(y_web_test, y_pred_rf),
        accuracy_score(y_web_test, y_pred_knn),
        web_acc_sc, web_acc_ft
    ],
    "Recall (Phishing)": [
        float(bl_lr_r['1']['recall']),
        float(bl_rf_r['1']['recall']),
        float(bl_knn_r['1']['recall']),
        float(sc_r['1']['recall']),
        float(ft_r['1']['recall'])
    ],
    "F1-score (Phishing)": [
        float(bl_lr_r['1']['f1-score']),
        float(bl_rf_r['1']['f1-score']),
        float(bl_knn_r['1']['f1-score']),
        float(sc_r['1']['f1-score']),
        float(ft_r['1']['f1-score'])
    ],
    "ROC-AUC": [
        roc_auc_score(y_web_test, y_prob_lr),
        roc_auc_score(y_web_test, y_prob_rf),
        roc_auc_score(y_web_test, y_prob_knn),
        web_roc_sc, web_roc_ft
    ]
}).set_index("Model")

comparison_pct = (comparison * 100).round(1)
display(comparison_pct.style.format("{:.1f}%"))

In [ ]:
# Heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(comparison_pct, annot=True, fmt=".1f", cmap="RdYlGn", linewidths=0.5)
plt.title("MLP Transfer Learning vs All Baselines (Web Page Dataset)")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
metrics = ['Accuracy', 'Recall (Phishing)', 'F1-score (Phishing)', 'ROC-AUC']

for idx, metric in enumerate(metrics):
    vals = comparison_pct[metric].values
    colors = ['#85C1E9', '#85C1E9', '#85C1E9', '#F5B041', '#7D3C98']
    axes[idx].bar(range(len(vals)), vals, color=colors, edgecolor='#333')
    axes[idx].set_ylabel(f'{metric} (%)')
    axes[idx].set_title(metric)
    axes[idx].set_xticks(range(len(vals)))
    axes[idx].set_xticklabels(['LR', 'RF', 'KNN', 'MLP\n(scratch)', 'MLP\n(transfer)'], fontsize=9)
    axes[idx].set_ylim(0, 105)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.suptitle('MLP Transfer Learning vs All Baselines - Web Page Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2.10 Confusion Matrices - MLP Scratch vs Transfer


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_sc = confusion_matrix(y_web_test, web_preds_sc)
sns.heatmap(cm_sc, annot=True, fmt='d', cmap='Oranges', ax=axes[0],
            xticklabels=['Legitimate (0)', 'Phishing (1)'],
            yticklabels=['Legitimate (0)', 'Phishing (1)'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('MLP - No Transfer (From Scratch)')

cm_ft = confusion_matrix(y_web_test, web_preds_ft)
sns.heatmap(cm_ft, annot=True, fmt='d', cmap='Purples', ax=axes[1],
            xticklabels=['Legitimate (0)', 'Phishing (1)'],
            yticklabels=['Legitimate (0)', 'Phishing (1)'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_title('MLP - With Transfer Learning')

plt.suptitle('MLP: Scratch vs Transfer Learning', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.2.12 Approach 2 Summary

**TOR Alignment:** This approach directly implements "pre-trained on one dataset and fine-tuned on the other" using PyTorch (as permitted by TOR).

**Pre-training (Phase 1):** MLP trained on URL numerical features (16 features in shared space of 27).

**Fine-tuning (Phase 2):** Same MLP fine-tuned on Web Page features (19 features in shared space of 27). Learning rate matched with the from-scratch baseline (0.001) so the comparison isolates the effect of pre-training.

**Fair Comparison:** All models evaluated on the **same** Web Page test set (from Week 1 split). The difference between "MLP from scratch" and "MLP transfer learning" shows the direct impact of cross-dataset knowledge transfer.

**Shared Feature Space:** 27 unified features (16 URL + 19 Web Page − 8 shared after column-name alignment: `url_length`, `n_dots`, `n_hypens`, `n_underline`, `n_slash`, `n_questionmark`, `n_and`, `n_at`). Missing features filled with 0, then StandardScaler applied.

**Key Finding:** MLP (both with and without transfer) outperformed all traditional baselines (81.3-81.7% accuracy vs 74.4-77.0%), showing the value of neural network approaches. The from-scratch MLP slightly edges out the transfer version on overall accuracy (81.7% vs 81.3%) and ROC-AUC (88.5% vs 88.4%), while the transfer version achieves marginally higher Phishing Recall (91.8% vs 91.1%) - a small precision-recall trade-off discussed further in the multi-seed validation (Section 5.2 Extension).

**This is the primary proposed model** for the project.


---

# Section 5.2 Extension - Multi-Seed Statistical Validation of Approach 2

## Purpose of This Experiment

The Week 5 results reported earlier in Section 5.2 compared **MLP-Transfer (81.3%)** against **MLP-No-Transfer (81.7%)** using a **single random seed (`random_state=42`)**. The from-scratch MLP slightly edged out the transfer version on overall accuracy (gap of about **−0.37 percentage points**) and on ROC-AUC, while the transfer version achieved a small **+0.78 percentage point gain on phishing recall**. A single-seed comparison cannot rigorously distinguish a **true effect** from **run-to-run variance** caused by random weight initialisation, mini-batch shuffling, and dropout stochasticity, so this small precision-recall trade-off needs a sanity-check across seeds before being trusted.

**Note on scope:** This extension cell is not a new work-package from the Terms of Reference. It is a statistical-rigour supplement to the Approach 2 experiment (originally Week 5 in the TOR Project Plan). It replaces no prior work and adds no new methodology - it simply repeats the same three-phase pipeline with four additional random seeds so that the single-run findings can be reported with statistical confidence intervals rather than as point estimates.

## Experimental Design

To address the single-seed limitation and meet the KV6013 Marking Scheme's requirement for **"level of confidence in the student's findings"** and **"how far the results can be generalised"** (Evaluation section, 80%+ band), the Approach 2 experiment is repeated across **5 independent random seeds: `[42, 7, 13, 2024, 99]`**.

For each seed, the complete three-phase pipeline from Section 5.2 is re-executed:

1. **Phase 1 - Pre-training** the MLP on the URL dataset (30 epochs, lr = 1×10⁻³)
2. **Phase 2 - Fine-tuning** the pre-trained MLP on the Web Page dataset (30 epochs, lr = 1×10⁻³, matched with Phase 3 for a strict fair comparison) → *MLP-Transfer*
3. **Phase 3 - Training from scratch** on the Web Page dataset with identical architecture but random initialisation (30 epochs, lr = 1×10⁻³) → *MLP-No-Transfer* (control)

## Statistical Analysis

After all five seeds complete, the results are aggregated as:

- **Mean ± standard deviation** for each metric (accuracy, ROC-AUC, phishing recall, phishing F1)
- **Paired t-test** between Transfer and No-Transfer across the 5 seeds (each seed provides one paired observation)
- **p-values** to determine statistical significance of observed differences

## Expected Runtime

On Google Colab Pro with an NVIDIA A100 GPU, the full five-seed experiment takes approximately **15-20 minutes** (≈3 minutes per seed × 5 seeds, with each seed running 90 epochs total across the three phases).

## Why This Matters

The multi-seed run produces a sharper picture than the single-seed run could give. Across five seeds, MLP-Transfer averaged **81.16% accuracy** and MLP-No-Transfer averaged **81.66%** - so the small accuracy disadvantage seen in the single-seed run holds up across seeds and is statistically significant (paired t-test, **p = 0.0054**). The same direction holds for ROC-AUC (Δ = **−0.28pp**, **p = 0.0499**), confirming that the mild negative transfer effect on these two metrics is genuine rather than a single-seed artefact.

The phishing recall picture, however, is more nuanced. Although MLP-Transfer achieved a higher mean recall (**+0.89 ± 2.28pp**), the direction was **not consistent across seeds** - three of the five seeds favoured MLP-No-Transfer (seeds 42, 7, 13) and two favoured MLP-Transfer (seeds 2024, 99) - and the paired t-test did not reach significance (**p = 0.4332**). The wide standard deviation (±2.28pp, larger than the mean itself) shows that the apparent recall gain is dominated by run-to-run variance rather than a robust transfer effect. This means the +0.78pp single-seed recall advantage observed in Section 5.2 cannot be confidently attributed to transfer learning.

This is exactly why the multi-seed validation is included: it converts Section 5.2 from a single-run observation that could be challenged in the viva into a finding with quantified confidence intervals and proper p-values - directly supporting the *"well-supported conclusions"* criterion in the Marking Scheme's 80%+ Evaluation band, including the discipline to report findings honestly even when the headline effect is small or null.

**Prerequisites (must already be in memory from Section 5.2):**
- `PhishingMLP` class (Section 5.2.2)
- `train_model`, `evaluate_model`, `create_loaders` helper functions (Section 5.2.3)
- Scaled 27-dimensional feature arrays: `X_url_tr_s`, `X_url_te_s`, `X_web_tr_s`, `X_web_te_s` (Section 5.2.1)
- Labels: `y_url_train`, `y_url_test`, `y_web_train`, `y_web_test`
- `input_dim` (= 27), `device` (= `cuda`)

In [ ]:
import numpy as np
import torch
from sklearn.metrics import classification_report
from scipy.stats import ttest_rel
import pandas as pd

# 5 random seeds for multi-run statistical validation
SEEDS = [42, 7, 13, 2024, 99]

# Storage for results
results = {
    'seed':         [],
    'transfer_acc': [], 'transfer_roc': [], 'transfer_recall': [], 'transfer_f1': [],
    'scratch_acc':  [], 'scratch_roc':  [], 'scratch_recall':  [], 'scratch_f1':  [],
}

print("MULTI-SEED VALIDATION OF TRANSFER EFFECT")
print(f"Running {len(SEEDS)} seeds: {SEEDS}")
print(f"Each seed: Phase 1 (30 ep) + Phase 2 Transfer (30 ep) + Phase 3 Scratch (30 ep)")

for seed_idx, seed in enumerate(SEEDS, 1):
    print(f"\n{'─' * 70}")
    print(f"SEED {seed_idx}/{len(SEEDS)}: random_state = {seed}")
    print(f"{'─' * 70}")

    # Set all random seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    # --- PHASE 1: Pre-train MLP on URL dataset ---
    print(f"\n[Seed {seed}] Phase 1: Pre-training on URL dataset...")
    url_train_loader, url_test_loader = create_loaders(
        X_url_tr_s, y_url_train, X_url_te_s, y_url_test, batch_size=256
    )
    model_pt = PhishingMLP(input_dim).to(device)
    train_model(model_pt, url_train_loader, url_test_loader,
                epochs=30, lr=0.001, device=device)

    # --- PHASE 2: Fine-tune on Web Page (TRANSFER) ---
    print(f"\n[Seed {seed}] Phase 2: Fine-tuning (Transfer)...")
    web_train_loader, web_test_loader = create_loaders(
        X_web_tr_s, y_web_train, X_web_te_s, y_web_test, batch_size=256
    )
    # Keep pre-trained weights, fine-tune with matched lr (Fix #E: lr=0.001 same as scratch baseline)
    train_model(model_pt, web_train_loader, web_test_loader,
                epochs=30, lr=0.001, device=device)   # matched LR (Fix #E)
    acc_t, roc_t, preds_t, probs_t = evaluate_model(
        model_pt, X_web_te_s, y_web_test, device
    )
    rep_t = classification_report(y_web_test, preds_t, output_dict=True)
    recall_t = float(rep_t['1']['recall'])
    f1_t = float(rep_t['1']['f1-score'])

    # --- PHASE 3: Train from scratch (NO TRANSFER - control) ---
    print(f"\n[Seed {seed}] Phase 3: Training from scratch (control)...")
    # Re-seed for fair comparison (same random init target, different starting point)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    model_sc = PhishingMLP(input_dim).to(device)
    train_model(model_sc, web_train_loader, web_test_loader,
                epochs=30, lr=0.001, device=device)
    acc_s, roc_s, preds_s, probs_s = evaluate_model(
        model_sc, X_web_te_s, y_web_test, device
    )
    rep_s = classification_report(y_web_test, preds_s, output_dict=True)
    recall_s = float(rep_s['1']['recall'])
    f1_s = float(rep_s['1']['f1-score'])

    # Record results
    results['seed'].append(seed)
    results['transfer_acc'].append(acc_t)
    results['transfer_roc'].append(roc_t)
    results['transfer_recall'].append(recall_t)
    results['transfer_f1'].append(f1_t)
    results['scratch_acc'].append(acc_s)
    results['scratch_roc'].append(roc_s)
    results['scratch_recall'].append(recall_s)
    results['scratch_f1'].append(f1_s)

    print(f"\n[Seed {seed}] Results:")
    print(f"  Transfer: acc={acc_t:.4f}, ROC={roc_t:.4f}, recall={recall_t:.4f}, F1={f1_t:.4f}")
    print(f"  Scratch:  acc={acc_s:.4f}, ROC={roc_s:.4f}, recall={recall_s:.4f}, F1={f1_s:.4f}")
    print(f"  Delta:    acc={acc_t-acc_s:+.4f}, ROC={roc_t-roc_s:+.4f}, recall={recall_t-recall_s:+.4f}")

# FINAL AGGREGATED RESULTS

df = pd.DataFrame(results)

print("\n\n" + "=" * 70)
print("MULTI-SEED RESULTS SUMMARY")
print(df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print("AGGREGATE STATISTICS (Mean ± Standard Deviation)")

print(f"\n{'Metric':<25} {'MLP-Transfer':<25} {'MLP-Scratch':<25} {'Delta':<15}")

metrics = [
    ('Accuracy',        'transfer_acc',    'scratch_acc'),
    ('ROC-AUC',         'transfer_roc',    'scratch_roc'),
    ('Phishing Recall', 'transfer_recall', 'scratch_recall'),
    ('Phishing F1',     'transfer_f1',     'scratch_f1'),
]

for metric_name, t_col, s_col in metrics:
    t_mean = np.mean(results[t_col])
    t_std = np.std(results[t_col], ddof=1)
    s_mean = np.mean(results[s_col])
    s_std = np.std(results[s_col], ddof=1)
    delta_mean = t_mean - s_mean
    delta_std = np.std(np.array(results[t_col]) - np.array(results[s_col]), ddof=1)
    print(f"{metric_name:<25} {t_mean:.4f} ± {t_std:.4f}      "
          f"{s_mean:.4f} ± {s_std:.4f}      {delta_mean:+.4f} ± {delta_std:.4f}")

# STATISTICAL SIGNIFICANCE TESTS (Paired t-tests)

print("PAIRED T-TESTS (Transfer vs Scratch, same seeds)")

for metric_name, t_col, s_col in metrics:
    t_arr = np.array(results[t_col])
    s_arr = np.array(results[s_col])
    t_stat, p_val = ttest_rel(t_arr, s_arr)
    sig = "***" if p_val < 0.001 else ("**" if p_val < 0.01 else ("*" if p_val < 0.05 else "n.s."))
    direction = "Transfer > Scratch" if t_stat > 0 else "Transfer < Scratch"
    print(f"  {metric_name:<20} t = {t_stat:+.3f}, p = {p_val:.4f}  [{sig}]  ({direction})")

print("\n  Significance: *** p<0.001, ** p<0.01, * p<0.05, n.s. = not significant")

# Save results to CSV for your records
df.to_csv('/content/multi_seed_results.csv', index=False)
print("\n Results saved to /content/multi_seed_results.csv")

---

## 5.3 Approach 3: Pseudo-Text Generation + DistilBERT Fine-tuning

### Pre-trained DistilBERT (Week 4) → Fine-tuned on Web Page Dataset via Pseudo-Text

This approach addresses the core challenge identified in Approach 1: DistilBERT requires text input, but the Web Page dataset contains only numerical features. Instead of concatenating embeddings (Approach 1) or switching to a different model (Approach 2), this approach **converts the Web Page numerical features into pseudo-text descriptions** that DistilBERT can process directly.

**Key idea:** Each row of numerical features is converted into a natural-language-style description (e.g., `"url_length 37 dots 3 hyphens 0 slash 0 redirection 0"`), enabling the pre-trained DistilBERT to apply its learned phishing detection patterns to the Web Page dataset.

**TOR Alignment:** This directly implements "fine-tuned on the second dataset" using the same DistilBERT model from Week 4, achieving true cross-dataset transfer learning.

This approach is presented as **comparative analysis** alongside Approach 1.

In [ ]:

import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


### 5.3.1 Convert Web Page Numerical Features to Pseudo-Text

Each row in the Web Page dataset is converted into a text string that describes its features.
For example, a row with `url_length=37, n_dots=3, n_hypens=0` becomes:
`"url_length 37 dots 3 hyphens 0 slash 0 questionmark 0 equal 0 at 0 and 0 exclamation 0 space 0 tilde 0 comma 0 plus 0 asterisk 0 hashtag 0 dollar 0 percent 0 redirection 0"`

This allows DistilBERT to process the Web Page data as text input, enabling direct fine-tuning.


In [ ]:
# Feature name mapping (shorter, readable names for pseudo-text)
feature_names = {
    'url_length': 'url_length',
    'n_dots': 'dots',
    'n_hypens': 'hyphens',
    'n_underline': 'underline',
    'n_slash': 'slash',
    'n_questionmark': 'questionmark',
    'n_equal': 'equal',
    'n_at': 'at',
    'n_and': 'and',
    'n_exclamation': 'exclamation',
    'n_space': 'space',
    'n_tilde': 'tilde',
    'n_comma': 'comma',
    'n_plus': 'plus',
    'n_asterisk': 'asterisk',
    'n_hastag': 'hashtag',
    'n_dollar': 'dollar',
    'n_percent': 'percent',
    'n_redirection': 'redirection'
}

def row_to_pseudo_text(row):
    """Convert a row of numerical features into pseudo-text for DistilBERT."""
    parts = []
    for col, name in feature_names.items():
        parts.append(f"{name} {int(row[col])}")
    return " ".join(parts)

# Apply to the cleaned Web Page dataset
feature_cols = [col for col in df_web_clean.columns if col != 'phishing']
df_web_clean['pseudo_text'] = df_web_clean.apply(row_to_pseudo_text, axis=1)

# Show examples
print(" Pseudo-text generation complete!")
print(f"   Total samples: {len(df_web_clean)}")
print(f"\nExample pseudo-text (first 3 rows):")
for i in range(3):
    label = "Phishing" if df_web_clean.iloc[i]['phishing'] == 1 else "Legitimate"
    print(f"\n  [{label}]: {df_web_clean.iloc[i]['pseudo_text']}")


### 5.3.2 Prepare Train/Test Split

Using the **same** stratified 80/20 split as all previous experiments for fair comparison.


In [ ]:
X_pseudo_text = df_web_clean['pseudo_text'].values
y_pseudo_labels = df_web_clean['phishing'].values

# Use same split as Week 1 (random_state=42, test_size=0.2, stratified)
X_train_pseudo, X_test_pseudo, y_train_pseudo, y_test_pseudo = train_test_split(
    X_pseudo_text, y_pseudo_labels,
    test_size=0.2,
    random_state=42,
    stratify=y_pseudo_labels
)

# Further split training into train and validation
X_train_pseudo, X_val_pseudo, y_train_pseudo, y_val_pseudo = train_test_split(
    X_train_pseudo, y_train_pseudo,
    test_size=0.1,
    random_state=42,
    stratify=y_train_pseudo
)

print(f"Training set:   {len(X_train_pseudo)} samples")
print(f"Validation set: {len(X_val_pseudo)} samples")
print(f"Test set:       {len(X_test_pseudo)} samples")
print(f"\nTraining label distribution:")
print(f"  Legitimate (0): {sum(y_train_pseudo == 0)} ({sum(y_train_pseudo == 0)/len(y_train_pseudo)*100:.1f}%)")
print(f"  Phishing (1):   {sum(y_train_pseudo == 1)} ({sum(y_train_pseudo == 1)/len(y_train_pseudo)*100:.1f}%)")


### 5.3.3 Load Pre-trained DistilBERT from Week 4

The same DistilBERT model that was fine-tuned on URL text in Week 4 is loaded. This model already understands phishing patterns from URL text - now we fine-tune it further on the Web Page pseudo-text.


In [ ]:
# Load the fine-tuned DistilBERT from Week 4
tokenizer = DistilBertTokenizer.from_pretrained('/content/distilbert_url_pretrained')
model_pseudo = DistilBertForSequenceClassification.from_pretrained('/content/distilbert_url_pretrained')
model_pseudo.to(device)

print(" Pre-trained DistilBERT loaded from Week 4!")
print(f"   Total parameters: {sum(p.numel() for p in model_pseudo.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in model_pseudo.parameters() if p.requires_grad):,}")


### 5.3.4 Tokenize Pseudo-Text

DistilBERT's tokenizer converts each pseudo-text string into token IDs. Since the pseudo-text is structured and shorter than full URLs, a max_length of 128 tokens is sufficient.

In [ ]:
# Show how pseudo-text is tokenized
sample_text = X_train_pseudo[0]
sample_tokens = tokenizer(sample_text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

print("Sample pseudo-text:", sample_text)
print(f"\nToken IDs (first 30): {sample_tokens['input_ids'][0][:30].tolist()}")
print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(sample_tokens['input_ids'][0][:30])}")
print(f"Total tokens used: {sample_tokens['attention_mask'][0].sum().item()} / 128")

### 5.3.5 Create PyTorch Dataset and DataLoaders


In [ ]:
class PseudoTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset_pseudo = PseudoTextDataset(X_train_pseudo, y_train_pseudo, tokenizer)
val_dataset_pseudo = PseudoTextDataset(X_val_pseudo, y_val_pseudo, tokenizer)
test_dataset_pseudo = PseudoTextDataset(X_test_pseudo, y_test_pseudo, tokenizer)

# Create DataLoaders
BATCH_SIZE = 32

train_loader_pseudo = DataLoader(train_dataset_pseudo, batch_size=BATCH_SIZE, shuffle=True)
val_loader_pseudo = DataLoader(val_dataset_pseudo, batch_size=BATCH_SIZE, shuffle=False)
test_loader_pseudo = DataLoader(test_dataset_pseudo, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training batches:   {len(train_loader_pseudo)}")
print(f"Validation batches: {len(val_loader_pseudo)}")
print(f"Test batches:       {len(test_loader_pseudo)}")


### 5.3.6 Fine-tune DistilBERT on Web Page Pseudo-Text

The pre-trained DistilBERT is fine-tuned on the Web Page pseudo-text with a **lower learning rate** (1e-5 vs 2e-5 used in Week 4) to preserve the phishing detection patterns learned from URLs while adapting to the new pseudo-text format.


In [ ]:
def train_epoch_pseudo(model, loader, optimizer, scaler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 50 == 0:
            print(f"    Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f} | Acc: {correct/total:.4f}")

    return total_loss / len(loader), correct / total


def evaluate_pseudo(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()

            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels, all_probs


In [ ]:
# Fine-tuning with lower learning rate to preserve URL knowledge
optimizer_pseudo = AdamW(model_pseudo.parameters(), lr=1e-5, weight_decay=0.01)
scaler_pseudo = GradScaler()

EPOCHS = 3

train_losses_p = []
val_losses_p = []
train_accs_p = []
val_accs_p = []

print("APPROACH 3: Fine-tuning DistilBERT on Web Page Pseudo-Text")
print(f"  Training samples: {len(X_train_pseudo)}")
print(f"  Using pre-trained weights from Week 4 (URL dataset)")
print(f"  Learning rate: 1e-5 (lower to preserve learned patterns)")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # Train
    train_loss, train_acc = train_epoch_pseudo(
        model_pseudo, train_loader_pseudo, optimizer_pseudo, scaler_pseudo, device
    )
    train_losses_p.append(train_loss)
    train_accs_p.append(train_acc)

    # Validate
    val_loss, val_acc, _, _, _ = evaluate_pseudo(model_pseudo, val_loader_pseudo, device)
    val_losses_p.append(val_loss)
    val_accs_p.append(val_acc)

    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

print(" Fine-tuning Complete!")


### 5.3.7 Training Progress Visualisation


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(range(1, EPOCHS+1), train_losses_p, 'b-o', label='Train Loss')
ax1.plot(range(1, EPOCHS+1), val_losses_p, 'r-o', label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Fine-tuning Loss (Pseudo-Text)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(range(1, EPOCHS+1), train_accs_p, 'b-o', label='Train Accuracy')
ax2.plot(range(1, EPOCHS+1), val_accs_p, 'r-o', label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Fine-tuning Accuracy (Pseudo-Text)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Approach 3: DistilBERT Fine-tuning on Web Page Pseudo-Text',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.3.8 Evaluate on Test Set


In [ ]:

test_loss_p, test_acc_p, test_preds_p, test_labels_p, test_probs_p = evaluate_pseudo(
    model_pseudo, test_loader_pseudo, device
)

print(" Approach 3: DistilBERT + Pseudo-Text - Test Results")
print(f"Accuracy: {test_acc_p:.4f}")
print(f"ROC-AUC:  {roc_auc_score(test_labels_p, test_probs_p):.4f}")
print(f"\nClassification Report:")
print(classification_report(
    test_labels_p, test_preds_p,
    target_names=['Legitimate (0)', 'Phishing (1)']
))

### 5.3.9 Confusion Matrix


In [ ]:
cm_pseudo = confusion_matrix(test_labels_p, test_preds_p)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_pseudo, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Legitimate (0)', 'Phishing (1)'],
            yticklabels=['Legitimate (0)', 'Phishing (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Approach 3 (DistilBERT + Pseudo-Text)')
plt.tight_layout()
plt.show()



---

# Week 6 - Model Testing and Performance Evaluation

## 6.1 Full Comparison - All Approaches on Web Page Dataset

This section consolidates the evaluation results from all approaches, comparing their performance on the **same** Web Page test set using the five TOR-required metrics: accuracy, precision, recall, F1-score, and ROC-AUC.

Models compared:
- **Traditional ML Baselines** (Week 2): LR, RF, KNN on numerical features
- **Approach 1** (Week 5): DistilBERT embeddings + Web Page numerical features
- **Approach 2** (Week 5): MLP Transfer Learning (pre-train URL → fine-tune Web Page) - **Primary Model**
- **Approach 3** (Week 5): DistilBERT fine-tuned on Web Page pseudo-text

In [ ]:

# Approach 3 metrics
pseudo_report = classification_report(test_labels_p, test_preds_p, output_dict=True)

# Build full comparison (using existing variables from previous cells)
bl_lr_r = classification_report(y_web_test, y_pred_lr, output_dict=True)
bl_rf_r = classification_report(y_web_test, y_pred_rf, output_dict=True)
bl_knn_r = classification_report(y_web_test, y_pred_knn, output_dict=True)
sc_r = classification_report(y_web_test, web_preds_sc, output_dict=True)
ft_r = classification_report(y_web_test, web_preds_ft, output_dict=True)

# Approach 1 metrics
tl_lr_r = classification_report(y_test_combined, y_pred_lr_comb, output_dict=True)
tl_rf_r = classification_report(y_test_combined, y_pred_rf_comb, output_dict=True)
tl_knn_r = classification_report(y_test_combined, y_pred_knn_comb, output_dict=True)

comparison_all = pd.DataFrame({
    "Model": [
        "LR (Baseline)", "RF (Baseline)", "KNN (Baseline)",
        "LR + DistilBERT Emb (Ap.1)", "RF + DistilBERT Emb (Ap.1)", "KNN + DistilBERT Emb (Ap.1)",
        "MLP No Transfer (Ap.2)", "MLP Transfer (Ap.2)",
        "DistilBERT Pseudo-Text (Ap.3)"
    ],
    "Accuracy": [
        accuracy_score(y_web_test, y_pred_lr),
        accuracy_score(y_web_test, y_pred_rf),
        accuracy_score(y_web_test, y_pred_knn),
        accuracy_score(y_test_combined, y_pred_lr_comb),
        accuracy_score(y_test_combined, y_pred_rf_comb),
        accuracy_score(y_test_combined, y_pred_knn_comb),
        web_acc_sc, web_acc_ft,
        test_acc_p
    ],
    "Recall (Phishing)": [
        float(bl_lr_r['1']['recall']),
        float(bl_rf_r['1']['recall']),
        float(bl_knn_r['1']['recall']),
        float(tl_lr_r['1']['recall']),
        float(tl_rf_r['1']['recall']),
        float(tl_knn_r['1']['recall']),
        float(sc_r['1']['recall']),
        float(ft_r['1']['recall']),
        float(pseudo_report['1']['recall'])
    ],
    "F1-score (Phishing)": [
        float(bl_lr_r['1']['f1-score']),
        float(bl_rf_r['1']['f1-score']),
        float(bl_knn_r['1']['f1-score']),
        float(tl_lr_r['1']['f1-score']),
        float(tl_rf_r['1']['f1-score']),
        float(tl_knn_r['1']['f1-score']),
        float(sc_r['1']['f1-score']),
        float(ft_r['1']['f1-score']),
        float(pseudo_report['1']['f1-score'])
    ],
    "ROC-AUC": [
        roc_auc_score(y_web_test, y_prob_lr),
        roc_auc_score(y_web_test, y_prob_rf),
        roc_auc_score(y_web_test, y_prob_knn),
        roc_auc_score(y_test_combined, y_prob_lr_comb),
        roc_auc_score(y_test_combined, y_prob_rf_comb),
        roc_auc_score(y_test_combined, y_prob_knn_comb),
        web_roc_sc, web_roc_ft,
        roc_auc_score(test_labels_p, test_probs_p)
    ]
}).set_index("Model")

comparison_all_pct = (comparison_all * 100).round(1)
display(comparison_all_pct.style.format("{:.1f}%"))

# Heatmap
plt.figure(figsize=(12, 7))
sns.heatmap(comparison_all_pct, annot=True, fmt=".1f", cmap="RdYlGn", linewidths=0.5)
plt.title("All Approaches Comparison (Web Page Dataset) - Metrics (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()


### 6.2 Key Approaches Comparison (Bar Chart)

Comparing the best-performing model from each approach category to highlight the overall progression from traditional baselines to transfer learning approaches.

In [ ]:
# Compare key models: Best Baseline vs Approach 1 best vs Approach 2 vs Approach 3
key_models = pd.DataFrame({
    "Model": [
        "KNN (Best Baseline)",
        "RF + DistilBERT Emb\n(Approach 1)",
        "MLP Transfer\n(Approach 2)",
        "DistilBERT Pseudo-Text\n(Approach 3)"
    ],
    "Accuracy": [
        accuracy_score(y_web_test, y_pred_knn) * 100,
        accuracy_score(y_test_combined, y_pred_rf_comb) * 100,
        web_acc_ft * 100,
        test_acc_p * 100
    ],
    "Recall (Phishing)": [
        float(bl_knn_r['1']['recall']) * 100,
        float(tl_rf_r['1']['recall']) * 100,
        float(ft_r['1']['recall']) * 100,
        float(pseudo_report['1']['recall']) * 100
    ],
    "F1-score (Phishing)": [
        float(bl_knn_r['1']['f1-score']) * 100,
        float(tl_rf_r['1']['f1-score']) * 100,
        float(ft_r['1']['f1-score']) * 100,
        float(pseudo_report['1']['f1-score']) * 100
    ],
    "ROC-AUC": [
        roc_auc_score(y_web_test, y_prob_knn) * 100,
        roc_auc_score(y_test_combined, y_prob_rf_comb) * 100,
        web_roc_ft * 100,
        roc_auc_score(test_labels_p, test_probs_p) * 100
    ]
}).set_index("Model")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
colors = ['#2196F3', '#9C27B0', '#FF9800', '#4CAF50']
metrics = ['Accuracy', 'Recall (Phishing)', 'F1-score (Phishing)', 'ROC-AUC']

for idx, metric in enumerate(metrics):
    vals = key_models[metric].values
    bars = axes[idx].bar(range(len(vals)), vals, color=colors, edgecolor='black', linewidth=0.5)
    axes[idx].set_ylabel(f'{metric} (%)')
    axes[idx].set_title(metric, fontweight='bold')
    axes[idx].set_xticks(range(len(vals)))
    axes[idx].set_xticklabels(key_models.index, rotation=15, ha='right', fontsize=8)
    axes[idx].set_ylim(0, 105)
    axes[idx].grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, val in zip(bars, vals):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                      f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Key Approaches Comparison - Web Page Dataset',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

# Week 7 - Cross-Dataset Generalisation Analysis

## 7.1 Summary of Cross-Dataset Transfer Learning Findings

**Within-Dataset Transfer Learning (Week 4):** DistilBERT fine-tuned on the URL dataset achieves very high accuracy on the same test set as the URL baselines, showing that pre-trained language models can learn deep phishing patterns from raw URL text that manual feature extraction misses. On the URL test set, DistilBERT reaches **98.8% accuracy** and **95.6% phishing recall**, compared to the strongest baseline (Random Forest) at **90.9% accuracy** and **73.6% phishing recall**.

**Cross-Dataset Transfer Learning (Week 5):** Three approaches were explored to transfer knowledge from the URL dataset to the Web Page dataset:

- **Approach 1 (DistilBERT Embeddings):** Did not improve over baselines. The random pairing of URL embeddings with unrelated Web Page samples introduced noise rather than useful information. This shows that embedding concatenation fails when the source and target datasets have no shared samples.

- **Approach 2 (MLP Transfer Learning - Primary Model):** Uses a shared 27-dimensional numerical feature space between the two datasets. Eight feature names are now aligned (`url_length`, `n_dots`, `n_hypens`, `n_underline`, `n_slash`, `n_questionmark`, `n_and`, `n_at`), so weights learned during URL pre-training are reused - rather than discarded - when fine-tuning on Web Page data. In single-seed runs, MLP-Transfer achieves **81.3% accuracy** and **91.8% phishing recall**, while MLP-No-Transfer (control) achieves **81.7% accuracy** and **91.1% phishing recall** - a small precision-recall trade-off further validated across 5 seeds in Section 5.2 Extension.

- **Approach 3 (Pseudo-Text + DistilBERT):** A creative approach that converts Web Page numerical features into text descriptions and continues fine-tuning the Week-4 DistilBERT checkpoint. This bridges the text-vs-numbers gap and stands as a comparative analysis alongside Approaches 1 and 2.

**Cross-Architectural Consistency:** The comparative analysis across three transfer learning approaches with different architectures (Random Forest on combined DistilBERT embeddings + Web features, MLP via shared 27-dimensional feature space, and fine-tuned DistilBERT on pseudo-text) shows that the conclusions are not an artefact of a single architectural choice. This cross-architectural pattern is reinforced by the multi-seed statistical validation in Section 5.2 Extension (n=5 random seeds, paired t-test).

**Robustness Component:** The TOR aim refers to *robustness and resilience against modern phishing strategies*. This is addressed in **Week 8 (Section 8)** through dedicated adversarial evaluation against URL-modification evasion strategies based on Pillai et al. (2024). The pre-trained DistilBERT model from Week 4 is re-tested on perturbed phishing URLs that mimic real-world attacker techniques, and the resulting degradation in phishing detection rate quantifies the model's adversarial robustness.



---

# Week 8 - Adversarial Robustness Testing (Evasion-Style Patterns)

## 8.1 Motivation and Approach

This section covers the **Week 8 deliverable** specified in the Terms of Reference: *"Robustness testing (evasion-style patterns)"*. It also addresses the *robustness and resilience against modern phishing strategies* component of the TOR Aim (Section 2) and the *robustness against real-world variations and evasion behaviours noted by Ghafoor et al. (2025)* requirement in the TOR Methodology (Section 4).

**The robustness gap.** A model can score very high on clean test data - as DistilBERT did in Week 4 - yet still fail under active adversarial conditions. Real-world phishing URLs are not static: attackers continuously modify URLs to evade detection. A trustworthy phishing detector must therefore be evaluated not only on clean test data but also on **adversarially-perturbed** versions of the same URLs.

**Approach.** Following the evasion attack methodology of **Pillai et al. (2024)** - *"Evasion Attacks and Defense Mechanisms for Machine Learning-Based Web Phishing Classifiers"* (IEEE Access, vol. 12, pp. 19375-19387) - five lexical evasion transformations are applied to phishing URLs from the URL test set. The pre-trained DistilBERT model from Week 4 is then re-evaluated on each perturbed set, and the resulting **detection rate** (phishing recall under attack) is compared to the original baseline.

**Strategies tested** (designed against the *actual* signal distribution of this dataset, not generic templates):

1. **Keyword Homoglyph Substitution** - replace lookalike characters (`o`→`0`, `i`→`1`, `l`→`1`, `a`→`@`) **inside known phishing keywords only** (`login`→`l0g1n`, `paypal`→`p@yp@l`). This targets the high-signal vocabulary the model relies on (e.g. `login` appears in 14% of phishing URLs and 0% of legitimate URLs).
2. **Keyword URL-Encoding** - percent-encode phishing keywords (`login`→`%6C%6F%67%69%6E`). DistilBERT's WordPiece tokenizer fragments percent-encoded text into opaque sub-tokens, denying the model recognisable phishing vocabulary.
3. **Hyphen Insertion** - insert hyphens into the domain. Hyphens are a *legitimate* signal in this dataset (mean 1.33 in legitimate vs 0.61 in phishing), so injecting them shifts the URL toward the legitimate side.
4. **Path Truncation** - phishing URLs in this dataset are 53% longer on average; truncating the path reduces overall length and the number of suspicious tokens, mimicking the shorter shape of legitimate URLs.
5. **Dot Reduction** - replace dots in the path portion with hyphens. Lowers a phishing signal (dots: 2.77 phish vs 1.78 legit) while raising a legitimate signal (hyphens).

**Metric.** The primary metric is the **Phishing Detection Rate** = recall on phishing URLs. Higher = the model still detects phishing despite the attack. Lower = the model has been fooled. The *degradation* (original detection − evasion detection, in percentage points) quantifies the model's adversarial vulnerability for each strategy.

### 8.2 Define Evasion Transformations

Each transformation is grounded in the actual phishing-vs-legitimate signal distribution of this dataset, identified in the data-analysis phase of Week 1. The functions are deliberately deterministic when given a fixed `random.seed`, ensuring reproducibility.

In [ ]:
# Evasion transformations grounded in the actual phishing-vs-legitimate
# distribution of the URL dataset.  Each transformation is designed to hide
# phishing signals or introduce legitimate-style features - the actual
# direction of an evasion attack.
#
# Reference: Pillai et al. (2024), "Evasion Attacks and Defense Mechanisms
# for Machine Learning-Based Web Phishing Classifiers" (IEEE Access 12).
import random
import re

PHISH_KEYWORDS = [
    'login', 'paypal', 'webscr', 'secure', 'account', 'verify',
    'update', 'confirm', 'signin', 'bank', 'wallet', 'wp-content',
    'wp-admin', 'admin', 'service'
]

# Strategy 1 - Keyword homoglyph substitution
def evasion_keyword_homoglyph(url):
    """Substitute lookalike characters inside phishing keywords (login -> l0g1n)."""
    subs = {'o': '0', 'i': '1', 'l': '1', 'a': '@', 's': '$', 'e': '3'}
    out = url
    for kw in PHISH_KEYWORDS:
        if kw in out.lower():
            def repl(m):
                tok = m.group(0)
                return ''.join(subs.get(c.lower(), c) for c in tok)
            out = re.sub(re.escape(kw), repl, out, flags=re.IGNORECASE)
    return out

# Strategy 2 - Keyword URL-encoding
def evasion_keyword_encoding(url):
    """Percent-encode phishing keywords (login -> %6C%6F%67%69%6E)."""
    out = url
    for kw in PHISH_KEYWORDS:
        if kw in out.lower():
            encoded = ''.join(f"%{ord(c):02X}" for c in kw)
            out = re.sub(re.escape(kw), encoded, out, flags=re.IGNORECASE)
    return out

# Strategy 3 - Hyphen insertion in the domain (hyphens are a legitimate signal)
def evasion_hyphen_insertion(url, count=3):
    """Insert hyphens into the domain to mimic legitimate URL style."""
    if '://' in url:
        scheme_end = url.find('://') + 3
        rest = url[scheme_end:]
        prefix = url[:scheme_end]
    else:
        prefix = ''
        rest = url
    slash = rest.find('/')
    if slash == -1:
        domain, path = rest, ''
    else:
        domain, path = rest[:slash], rest[slash:]
    if len(domain) > 6:
        positions = list(range(2, len(domain) - 2))
        n = min(count, len(positions))
        if n > 0:
            chosen = sorted(random.sample(positions, n))
            for p in reversed(chosen):
                domain = domain[:p] + '-' + domain[p:]
    return prefix + domain + path

# Strategy 4 - Path truncation
def evasion_path_truncation(url, keep_chars=20):
    """Truncate URL path to mimic the shorter shape of legitimate URLs."""
    if '://' in url:
        scheme_end = url.find('://') + 3
        rest = url[scheme_end:]
        prefix = url[:scheme_end]
    else:
        prefix = ''
        rest = url
    slash = rest.find('/')
    if slash == -1:
        return url
    domain, path = rest[:slash], rest[slash:]
    if len(path) > keep_chars:
        path = path[:keep_chars]
    return prefix + domain + path

# Strategy 5 - Dot reduction
def evasion_dot_reduction(url):
    """Replace dots in the path portion with hyphens."""
    if '://' in url:
        scheme_end = url.find('://') + 3
        rest = url[scheme_end:]
        prefix = url[:scheme_end]
    else:
        prefix = ''
        rest = url
    slash = rest.find('/')
    if slash == -1:
        return url
    domain, path = rest[:slash], rest[slash:]
    path = path.replace('.', '-')
    return prefix + domain + path

print("5 evasion transformations defined, grounded in actual data distribution.")
print("Each transformation is designed to REDUCE phishing detection signals.")

### 8.3 Inspect Example Transformations

Three example phishing URLs are passed through every transformation so that the perturbations can be visually inspected. This sanity check confirms that each transformation produces realistic, attacker-style variations.

In [ ]:
# Pick 3 phishing URLs from the test set as examples
import pandas as pd

# Build a Series of phishing URLs from the Week-4 test set
test_url_series   = pd.Series(list(X_test_text))    if not isinstance(X_test_text,   pd.Series) else X_test_text.reset_index(drop=True)
test_label_series = pd.Series(list(y_test_labels))  if not isinstance(y_test_labels, pd.Series) else y_test_labels.reset_index(drop=True)

phishing_mask = test_label_series == 1
phishing_urls_pool = test_url_series[phishing_mask].reset_index(drop=True)
print(f"Phishing URLs available in test set: {len(phishing_urls_pool):,}")

# Display 3 examples per strategy
random.seed(42)
example_indices = random.sample(range(len(phishing_urls_pool)), 3)
example_urls = [phishing_urls_pool[i] for i in example_indices]

strategies = {
    'Original (no evasion)':  lambda x: x,
    'Keyword Homoglyph':      evasion_keyword_homoglyph,
    'Keyword URL-Encoding':   evasion_keyword_encoding,
    'Hyphen Insertion':       evasion_hyphen_insertion,
    'Path Truncation':        evasion_path_truncation,
    'Dot Reduction':          evasion_dot_reduction,
}

print('EXAMPLE EVASION TRANSFORMATIONS')
for url in example_urls:
    print(f"\nOriginal: {url}")
    for name, fn in strategies.items():
        if name == 'Original (no evasion)':
            continue
        random.seed(hash(url) % (2**31))
        print(f"  -> {name:22s}: {fn(url)}")

### 8.4 Load Pre-trained DistilBERT (Week 4 Checkpoint)

The model under test is the same DistilBERT checkpoint produced by Week 4 - this ensures the robustness test measures the actual deployed model. No further training takes place in Week 8; this is purely an evaluation phase.

In [ ]:
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer_robust = DistilBertTokenizer.from_pretrained('/content/distilbert_url_pretrained')
model_robust    = DistilBertForSequenceClassification.from_pretrained('/content/distilbert_url_pretrained')
model_robust.to(device)
model_robust.eval()

print(f"DistilBERT loaded from Week 4 checkpoint on {device}")
print(f"   Total parameters: {sum(p.numel() for p in model_robust.parameters()):,}")

### 8.5 Evaluate Phishing Detection Rate Under Each Evasion Strategy

A sample of 2,000 phishing URLs is drawn from the URL test set. Each URL is passed through every evasion transformation, and the perturbed batches are fed through the loaded DistilBERT model. The proportion of perturbed URLs still classified as phishing is reported as the *detection rate* under that attack.

In [ ]:
import numpy as np

SAMPLE_SIZE = 2000
random.seed(42)
sample_indices = random.sample(range(len(phishing_urls_pool)), SAMPLE_SIZE)
phishing_sample = [phishing_urls_pool[i] for i in sample_indices]
print(f"Sampled {len(phishing_sample):,} phishing URLs for robustness evaluation.")

def predict_phishing(urls, model, tokenizer, device, batch_size=32):
    """Return binary predictions (1 = phishing) for a list of URLs."""
    model.eval()
    all_preds = []
    for i in range(0, len(urls), batch_size):
        batch = urls[i:i + batch_size]
        enc = tokenizer(batch, truncation=True, padding=True, max_length=128, return_tensors='pt')
        input_ids = enc['input_ids'].to(device)
        attn_mask = enc['attention_mask'].to(device)
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attn_mask)
            preds = torch.argmax(out.logits, dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
    return np.array(all_preds)

# Run each strategy
robustness_results = {}
print("ROBUSTNESS RESULTS - Phishing Detection Rate Under Evasion")
for name, transform in strategies.items():
    random.seed(42)
    perturbed = [transform(u) for u in phishing_sample]
    preds = predict_phishing(perturbed, model_robust, tokenizer_robust, device)
    detection_rate = (preds == 1).mean() * 100
    robustness_results[name] = detection_rate
    print(f"  {name:25s} -> Detection: {detection_rate:6.2f}%")

# Compute degradation
baseline_rate = robustness_results['Original (no evasion)']
print("\nDEGRADATION FROM BASELINE")
for name, rate in robustness_results.items():
    if name == 'Original (no evasion)':
        continue
    drop = baseline_rate - rate
    print(f"  {name:25s} -> Drop: {drop:+6.2f} pp")

### 8.6 Visualise Robustness Results

The bar chart below compares detection rate on each evasion strategy against the original (unperturbed) baseline. The dashed line marks the baseline so the magnitude of degradation is visually obvious.

In [ ]:
import matplotlib.pyplot as plt

names  = list(robustness_results.keys())
rates  = [robustness_results[n] for n in names]
colors = ['#27AE60'] + ['#E74C3C', '#F39C12', '#9B59B6', '#3498DB', '#E67E22']

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.bar(names, rates, color=colors, edgecolor='black')
ax.set_ylabel('Phishing Detection Rate (%)', fontsize=12)
ax.set_title('Week 8 - DistilBERT Robustness Under Evasion Attacks (Pillai et al., 2024)',
             fontsize=13, fontweight='bold')
ax.set_ylim([0, 110])
ax.axhline(y=rates[0], color='gray', linestyle='--', alpha=0.6,
           label=f'Original baseline ({rates[0]:.1f}%)')
ax.legend(loc='lower right')
plt.xticks(rotation=15, ha='right')

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{rate:.1f}%', ha='center', fontsize=11, fontweight='bold')

for i, (bar, rate) in enumerate(zip(bars[1:], rates[1:]), start=1):
    drop = rates[0] - rate
    if rate > 8:
        ax.text(bar.get_x() + bar.get_width() / 2, rate / 2,
                f'{-drop:+.1f}pp', ha='center', color='white',
                fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('week8_robustness_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: week8_robustness_chart.png")

### 8.7 Confusion Matrix - Original vs Worst-Case Evasion

The two confusion matrices below contrast the model's performance on the 2,000 phishing URLs before and after the worst-performing evasion strategy. All ground-truth labels are phishing, so the bottom-left cell counts **false negatives** - phishing URLs the model failed to detect. Growth of this cell visualises the attack's success.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

worst_name, worst_rate = min(
    ((n, r) for n, r in robustness_results.items() if n != 'Original (no evasion)'),
    key=lambda kv: kv[1]
)
print(f"Worst-case evasion strategy: {worst_name} ({worst_rate:.2f}% detection)")

random.seed(42)
worst_perturbed = [strategies[worst_name](u) for u in phishing_sample]
worst_preds = predict_phishing(worst_perturbed, model_robust, tokenizer_robust, device)

random.seed(42)
original_preds = predict_phishing(phishing_sample, model_robust, tokenizer_robust, device)

y_true = np.ones(SAMPLE_SIZE, dtype=int)
cm_orig  = confusion_matrix(y_true, original_preds, labels=[0, 1])
cm_worst = confusion_matrix(y_true, worst_preds,    labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm_orig, annot=True, fmt='d', cmap='Greens', cbar=False,
            xticklabels=['Predicted\nLegitimate', 'Predicted\nPhishing'],
            yticklabels=['(none)', 'Actual\nPhishing'], ax=axes[0],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title(f'Original URLs\nDetection: {robustness_results["Original (no evasion)"]:.1f}%',
                  fontsize=12, fontweight='bold')

sns.heatmap(cm_worst, annot=True, fmt='d', cmap='Reds', cbar=False,
            xticklabels=['Predicted\nLegitimate', 'Predicted\nPhishing'],
            yticklabels=['(none)', 'Actual\nPhishing'], ax=axes[1],
            annot_kws={'size': 14, 'weight': 'bold'})
axes[1].set_title(f'After {worst_name}\nDetection: {worst_rate:.1f}%',
                  fontsize=12, fontweight='bold')

plt.suptitle('Phishing Detection: Original vs Worst-Case Evasion (n = 2,000 phishing URLs)',
             fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig('week8_robustness_confusion.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: week8_robustness_confusion.png")

### 8.8 Discussion and Robustness Findings

The evasion testing in this section measures the DistilBERT model's behaviour under **adversarial conditions** - the most meaningful test for any phishing detector destined for real-world deployment.

**Original baseline (no evasion).** Under unperturbed conditions, the model achieves a **95.50% phishing detection rate** on the 2,000-URL sample, in line with the recall reported in Week 4 and confirming that the sample is representative of the broader test set.

**Under evasion.** **All five** attacker-style transformations cause the detection rate to drop, but the magnitudes differ substantially. The largest drop occurs under **Keyword URL-Encoding (-3.65 percentage points)**, followed by Path Truncation (-3.00 pp), Hyphen Insertion (-1.55 pp), Dot Reduction (-1.30 pp), and Keyword Homoglyph (-0.60 pp). The bar chart in Section 8.6 shows the full distribution of changes.

**Why each strategy works on this dataset.**

- *Keyword Homoglyph* - The model has learned that tokens like `login`, `paypal`, and `webscr` strongly indicate phishing (`login` appears in 14% of phishing URLs versus 0% of legitimate URLs in the training data). Substituting characters inside these keywords (`login` → `l0g1n`) produces sub-token sequences the model never saw during pre-training, weakening the signal. The effect is mild here (-0.60 pp) because most phishing URLs in the test sample do not contain the targeted keywords, so the transformation reaches only a fraction of the sample.

- *Keyword URL-Encoding* - Percent-encoding (`login` → `%6C%6F%67%69%6E`) replaces a familiar word with a sequence of opaque sub-tokens. WordPiece tokenises the encoded form into nonsensical fragments, removing the recognisable phishing vocabulary. This is the strongest attack tested (-3.65 pp), confirming that obscuring known phishing keywords is more disruptive than simply substituting characters within them.

- *Hyphen Insertion* - Hyphens are a *legitimate* signal in this dataset by frequency (mean 1.33 in legitimate vs 0.61 in phishing), so the design intent of this attack is to shift the URL's lexical profile toward the legitimate side. The attack worked as designed but only modestly (-1.55 pp): DistilBERT does not rely on raw hyphen counts the way a feature-engineered model would, so simply injecting hyphens into the domain (e.g. `mo-bile-fa-cebook--com.com`) disrupts the model less than directly attacking its phishing vocabulary. This is itself an instructive robustness finding - hyphen frequency is a useful but secondary signal for the model.

- *Path Truncation* - Phishing URLs are 53% longer on average than legitimate ones. Truncating the path reduces overall length and the count of suspicious tokens (slashes, dots), again moving the URL into legitimate territory. This is the second-strongest attack (-3.00 pp) and confirms that URL length is a meaningful signal the model uses.

- *Dot Reduction* - Dots correlate with phishing (2.77 mean in phishing vs 1.78 in legitimate). Replacing dots in the path with hyphens lowers a phishing signal while raising a legitimate one. The mild impact (-1.30 pp) is consistent with the Hyphen Insertion result: the model is not heavily dependent on raw character-frequency signals.

**Key finding.** DistilBERT shows **moderate-to-strong robustness** to the five lexical evasion strategies tested. The worst-case degradation is **-3.65 percentage points** (Keyword URL-Encoding) and the mean degradation across the five attacks is approximately **-2.0 pp**. This is consistent with the broader observation that transformer-based detectors trained on raw URL text are harder to evade with simple character-level perturbations than feature-engineered classifiers (Pillai et al., 2024). The model nonetheless retains a measurable adversarial surface; deeper attacks (visual, content-level, or adaptive white-box) are out of scope for a URL-only detector and remain valid directions for future defensive work using techniques such as adversarial training, ensemble detection, or HTML/DOM features from the Web Page dataset.

**Why this matters for the TOR aim.** The TOR Aim (Section 2) commits to evaluating *robustness and resilience against modern phishing strategies*. The cross-architectural and multi-seed checks in Section 7.1 establish *consistency* of the transfer mechanism. This Week 8 evaluation establishes *adversarial robustness limits* - together, the two sets of evidence cover both senses of *robustness* used in the TOR.


### 8.9 Week 8 Summary

**TOR Alignment.** This section covers:

- TOR Section 2 (Aim) - *robustness and resilience against modern phishing strategies*
- TOR Section 3, Objective 5 - *analyse robustness against dataset shift* (complemented by Week 7)
- TOR Section 4 (Methodology) - *robustness against real-world variations and evasion behaviours noted by Ghafoor et al. (2025)*
- TOR Section 7, Project Plan **Week 8** - *Robustness testing (evasion-style patterns)*

**Methodology.** Five lexical evasion transformations grounded in the actual phishing-vs-legitimate distribution of this dataset were applied to a stratified sample of 2,000 phishing URLs from the URL test set. The pre-trained DistilBERT model from Week 4 was re-evaluated on each perturbed set, and the resulting detection rate was compared to the original baseline. No model retraining occurred; this is purely an evaluation phase.

**Outcome.** Detection-rate degradation under evasion was quantified, providing empirical evidence of the model's adversarial vulnerability surface.

**Limitations.** The five strategies tested cover **lexical-level URL evasion only**. Deeper attacks remain out of scope for this URL-based detector:

- Visual phishing (page screenshots that mimic legitimate sites)
- Content-level spoofing (HTML/DOM evasion handled by the Web Page dataset, not URL strings)
- Fast-flux DNS and IP-rotation attacks
- Adaptive (white-box) attacks where the attacker has gradient access to the model

**Position in the project.**

| Week | Robustness layer | Status |
|------|-----------------|--------|
| Week 4 | Within-dataset transfer (URL only) | done |
| Week 5 | Cross-dataset transfer (URL → Web Page) | done - 3 approaches |
| Section 5.2 Ext. | Multi-seed statistical reproducibility | done - 5 seeds, paired t-test |
| Week 7 | Cross-dataset generalisation analysis | done - heterogeneous datasets |
| **Week 8** | **Adversarial / evasion robustness** | **done - 5 strategies tested** |

With Week 8 in place, every robustness item listed in the TOR has a corresponding deliverable in the notebook.